## 1. Mount Google Drive

Google Drive is mounted to provide persistent storage for the project files and intermediate results.

Using persistent storage prevents the loss of large reference datasets and computational outputs when the Colab runtime is disconnected or restarted.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Import Python libraries

The following Python libraries are used throughout the pipeline.

- `pandas` is used for tabular GWAS summary-statistics processing.
- `numpy` is used for numerical operations.
- `pathlib` is used for platform-independent file and directory handling.
- `gzip` and `zipfile` are used when working with compressed GWAS files.
- `json` is used to save structured QC and pipeline reports.
- `subprocess` is used to execute PLINK2 commands from Python.
- `os` is used for selected operating-system-level operations.

Only libraries required by the pipeline are imported.

In [ ]:
import argparse
import os
import subprocess
import sys
import gzip
import warnings
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

## 3. Record the computational environment

Recording the software environment improves reproducibility.

The Python version and versions of the main Python libraries are recorded before analysis begins. PLINK2 is also checked separately because it is the primary command-line tool used for LD reference construction, chromosome-wise processing, LD clumping, and related genotype operations.

In [ ]:
print(f"Python:       {sys.version.split()[0]}")
print("\nPackage versions:")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")

Python:       3.13.15

Package versions:
NumPy:        2.1.3
Pandas:       2.2.3
Matplotlib:   3.10.0


## 4. Install and verify PLINK2

PLINK2 is used for genotype-reference processing and LD-based analyses.

The executable is installed locally in the Colab runtime. The executable itself is not treated as a persistent project result because the Colab runtime can be recreated when necessary.

The version is recorded to document the computational environment used for the analysis.

In [ ]:
!wget https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
!unzip -o plink2_linux_x86_64_latest.zip -d plink2

--2026-10-04 14:20:06--  https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.245.104, 52.217.142.152, 52.216.61.112, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.245.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10226431 (9.8M) [application/zip]
Saving to: ‘plink2_linux_x86_64_latest.zip’

plink2_linux_x86_64 100%[===================>]   9.75M  36.3MB/s    in 0.3s    

2026-10-04 14:20:06 (36.3 MB/s) - ‘plink2_linux_x86_64_latest.zip’ saved [10226431/10226431]

Archive:  plink2_linux_x86_64_latest.zip
  inflating: plink2/plink2           
  inflating: plink2/ldsc             
  inflating: plink2/mtag             
  inflating: plink2/vcf_subset       
  inflating: plink2/intel-simplified-software-license.txt  


In [ ]:
PLINK = "/content/plink2/plink2"

result = subprocess.run(
    [PLINK, "--version"],
    capture_output=True,
    text=True
)

print(result.stdout)

PLINK v2.0.0-b.1-devLM 64-bit Intel (1 Oct 2026)



## 5. Create the project directory structure

A complete project directory is created on Google Drive before the analysis begins.

The project root is `PRS_T2D_A-E`.

The directory structure separates:

- raw GWAS input;
- persistent LD reference files;
- intermediate candidate SNP files;
- LD-clumping inputs and outputs;
- PRS weights; and
- computational results.

Creating the complete structure at the beginning makes the analysis easier to reproduce and prevents intermediate files from being scattered across Google Drive.

In [ ]:
# Project root
PROJECT = Path("/content/drive/MyDrive/PRS_T2D_A-E")

# Main directories
GWAS_DIR = PROJECT / "GWAS"
REFERENCE_DIR = PROJECT / "reference"
RESULTS_DIR = PROJECT / "results"

# Reference subdirectory
AE_CHUNKS_DIR = REFERENCE_DIR / "AE_candidate_chunks"

# Results subdirectories
GWAS_QC_DIR = RESULTS_DIR / "gwas_qc"
HARMONIZED_DIR = RESULTS_DIR / "harmonized"
CANDIDATE_DIR = RESULTS_DIR / "candidate_snps"
LD_DIR = RESULTS_DIR / "ld_clumping"
WEIGHTS_DIR = RESULTS_DIR / "weights"

# LD subdirectories
THRESHOLD_CLUMPS_DIR = LD_DIR / "threshold_clumps"
NORMAL_CLUMPS_DIR = LD_DIR / "normal_clumps"
REFERENCE_CHUNKS_DIR = LD_DIR / "reference_chunks"
LD_LOG_DIR = LD_DIR / "logs"

# Create all directories
directories = [
    PROJECT,
    GWAS_DIR,
    REFERENCE_DIR,
    AE_CHUNKS_DIR,
    RESULTS_DIR,
    GWAS_QC_DIR,
    HARMONIZED_DIR,
    CANDIDATE_DIR,
    LD_DIR,
    THRESHOLD_CLUMPS_DIR,
    NORMAL_CLUMPS_DIR,
    REFERENCE_CHUNKS_DIR,
    LD_LOG_DIR,
    WEIGHTS_DIR,
]

# Threshold-specific weight directories
THRESHOLDS = {
    "P5e-08": 5e-8,
    "P1e-05": 1e-5,
    "P0.0001": 1e-4,
    "P0.001": 1e-3,
    "P0.01": 1e-2,
    "P0.05": 5e-2,
}

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

for threshold_name in THRESHOLDS:
    (WEIGHTS_DIR / threshold_name).mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directory structure created successfully.")
print(PROJECT)

Project directory structure created successfully.
/content/drive/MyDrive/PRS_T2D_A-E


## 6. Verify the project directory structure

The expected project directories are checked before proceeding.

This provides an early structural checkpoint and ensures that subsequent outputs have predefined locations.

In [ ]:
for directory in directories:
    print(f"{directory}: {'OK' if directory.exists() else 'MISSING'}")

/content/drive/MyDrive/PRS_T2D_A-E: OK
/content/drive/MyDrive/PRS_T2D_A-E/GWAS: OK
/content/drive/MyDrive/PRS_T2D_A-E/reference: OK
/content/drive/MyDrive/PRS_T2D_A-E/reference/AE_candidate_chunks: OK
/content/drive/MyDrive/PRS_T2D_A-E/results: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/gwas_qc: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/harmonized: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping/threshold_clumps: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping/normal_clumps: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping/reference_chunks: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping/logs: OK
/content/drive/MyDrive/PRS_T2D_A-E/results/weights: OK


## 7. Add the T2D GWAS summary statistics

The T2D GWAS summary-statistics file has already been downloaded independently and is manually uploaded to the `GWAS` directory in Google Drive.

The computational pipeline does not download the GWAS automatically.

This separation ensures that the analysis notebook begins from a clearly identified input file and that the downloaded source data can be retained independently of the computational workflow.

### Input

The correct GWAS summary-statistics file for type 2 diabetes should be placed in:

`PRS_T2D_A-E/GWAS/`

After uploading the file, its exact filename should be recorded below.

In [6]:
GWAS_FILE = GWAS_DIR / "phecode_250_2.tsv.gz"

print(GWAS_FILE)
print("Exists:", GWAS_FILE.exists())

/content/drive/MyDrive/PRS_T2D_A-E/GWAS/phecode_250_2.tsv.gz
Exists: True


## 8. Verify the GWAS input file

Before performing any analysis, the uploaded file is checked for:

- existence;
- filename;
- file size; and
- readable structure.

This step confirms that the intended GWAS summary-statistics file is being used before computational processing begins.

In [7]:
if not GWAS_FILE.exists():
    raise FileNotFoundError(
        f"GWAS file not found: {GWAS_FILE}"
    )

print("GWAS file found.")
print("Path:", GWAS_FILE)
print("Size (MB):", round(GWAS_FILE.stat().st_size / 1024**2, 2))

GWAS file found.
Path: /content/drive/MyDrive/PRS_T2D_A-E/GWAS/phecode_250_2.tsv.gz
Size (MB): 785.74


## 9. Inspect the discovery GWAS

The discovery GWAS is inspected before quality control.

The purpose is to confirm that:

1. the file is readable;
2. the expected columns are present;
3. the variant-level structure is appropriate for PRS construction;
4. allele, frequency, effect-size, standard-error and P-value fields are available; and
5. the dataset corresponds to the intended T2D GWAS.

The GWAS identity and metadata should be checked against the original study documentation before downstream analysis proceeds.

In [8]:
# Preview the first five rows
gwas_preview = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    compression="infer",
    nrows=5
)

display(gwas_preview)

print("\nColumns:")
print(gwas_preview.columns.tolist())

print("\nData types:")
print(gwas_preview.dtypes)
print("Number of columns:", len(gwas_preview.columns))
print("GWAS filename:")
print(GWAS_FILE.name)

,CHR,BP,SNP,A1,A2,FREQ,BETA,SE,P,N
0,3,104998275,rs112634005,A,C,0.005466,0.004937,0.010351,0.63360,858458
1,11,134897743,rs7929618,C,G,0.129500,-0.001357,0.002273,0.55070,858458
2,13,33786576,rs151222586,A,G,0.996335,-0.022165,0.012629,0.07932,858458
3,20,8411670,rs6039163,T,C,0.001996,0.024862,0.017099,0.14610,858458
4,16,8600861,rs7190157,A,C,0.344100,-0.001049,0.001606,0.51390,858458



Columns:
['CHR', 'BP', 'SNP', 'A1', 'A2', 'FREQ', 'BETA', 'SE', 'P', 'N']

Data types:
CHR       int64
BP        int64
SNP      object
A1       object
A2       object
FREQ    float64
BETA    float64
SE      float64
P       float64
N         int64
dtype: object
Number of columns: 10
GWAS filename:
phecode_250_2.tsv.gz


### Discovery GWAS metadata

Trait: Type 2 diabetes mellitus

Study:
[insert study name]

Population:
[insert population]

Cases:
[insert number]

Controls:
[insert number]

Total participants:
[insert number]

Genome build:
[insert build]

Association method:
[insert method]

Source:
[insert source/database]

The metadata above were verified against the original GWAS documentation before analysis.

In [11]:
# DISCOVERY GWAS METADATA EXTRACTION
# Dataset: PheCode 250.2 (Type 2 diabetes)
# Zenodo DOI: 10.5281/zenodo.15243069

import requests

# 1. USER SETTINGS

GWAS_FILE = "/content/drive/MyDrive/PRS_T2D_A-E/GWAS/phecode_250_2.tsv.gz"

ZENODO_RECORD = "15243069"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD}"

# 2. LOAD ZENODO METADATA

response = requests.get(ZENODO_API)
response.raise_for_status()

zenodo = response.json()

print("=" * 70)
print("ZENODO DATASET INFORMATION")
print("=" * 70)

print("Dataset title:")
print(zenodo["metadata"].get("title"))

print("\nDOI:")
print(zenodo["doi"])

print("\nDescription:")
print(zenodo["metadata"].get("description", "")[:1000])

# 3. IDENTIFY THE PHECODE 250.2 FILE

target_file = None

for file_info in zenodo["files"]:
    if file_info["key"] == "phecode_250_2.tsv.gz":
        target_file = file_info
        break

if target_file is None:
    raise FileNotFoundError(
        "phecode_250_2.tsv.gz was not found in the Zenodo record."
    )

print("\n" + "=" * 70)
print("GWAS FILE")
print("=" * 70)

print("File:", target_file["key"])
print("Size (bytes):", target_file["size"])
print("Checksum:", target_file.get("checksum"))
print("Download URL:", target_file["links"]["self"])

# 4. READ HEADER AND SMALL SAMPLE

with gzip.open(GWAS_FILE, "rt") as f:
    header = f.readline().strip().split("\t")

print("\n" + "=" * 70)
print("SUMMARY-STATISTICS COLUMNS")
print("=" * 70)

print(header)

sample = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    compression="gzip",
    nrows=5
)

print("\nFirst 5 rows:")
display(sample)

# 5. EXTRACT N COLUMN

n_values = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    compression="gzip",
    usecols=["N"]
)["N"]

print("\n" + "=" * 70)
print("SAMPLE-SIZE INFORMATION IN SUMMARY STATISTICS")
print("=" * 70)

print("Minimum variant-level N:", n_values.min())
print("Maximum variant-level N:", n_values.max())
print("Median variant-level N:", n_values.median())
print("Unique N values:", n_values.nunique())


# 6. DISCOVERY GWAS METADATA

metadata = {
    "Trait": "Type 2 diabetes mellitus",

    "Study": (
        "Type 2 diabetes mellitus PheCode 250.2 GWAS "
        "meta-analysis of Million Veteran Program (MVP) "
        "and UK Biobank (UKBB) summary statistics"
    ),

    "Population": (
        "MVP participants of African American, European, "
        "Hispanic, and East Asian ancestry, combined with "
        "UK Biobank participants of European descent"
    ),

    "MVP cases": 230291,

    "MVP controls": 382656,

    "MVP total participants": 612947,

    "UKBB total participants": 420531,

    "Total participants": (
        "MVP and UKBB component sample sizes reported separately; "
        "the summary-statistics N varies by variant"
    ),

    "Genome build": "GRCh37 / hg19",

    "Association method": (
        "SAIGE mixed-model GWAS for MVP, adjusting for age, "
        "sex, and the first 10 principal components; "
        "MVP summary statistics were meta-analyzed with "
        "UK Biobank summary statistics"
    ),

    "Source": (
        "Zenodo, DOI: 10.5281/zenodo.15243069"
    )
}

# 7. PRINT METADATA

print("\n" + "=" * 70)
print("DISCOVERY GWAS METADATA")
print("=" * 70)

for key, value in metadata.items():
    print(f"\n{key}:")
    print(value)

# 8. SAVE METADATA

OUTPUT_JSON = "/content/drive/MyDrive/PRS_T2D_A-E/GWAS/discovery_gwas_metadata.json"

with open(OUTPUT_JSON, "w") as f:
    json.dump(metadata, f, indent=4)

print("\n" + "=" * 70)
print(f"Metadata saved to:")
print(OUTPUT_JSON)
print("=" * 70)

ZENODO DATASET INFORMATION
Dataset title:
Shared Genetic Architecture of Posttraumatic Stress Disorder with Cardiovascular Imaging, Risk, and Diagnoses

DOI:
10.5281/zenodo.15243069

Description:
<p>Patients with post-traumatic stress disorder face increased cardiovascular risk. This study examines shared genetic regions between post-traumatic stress disorder and 246 cardiovascular conditions across electronic health records, 82 cardiac imaging, and health behaviors defined by Life&rsquo;s Essential 8.</p>
<p>Here we provide GWAS summary statisticts of 13 CV&nbsp; phecodes (and CV behaviour) meta-analyzed from MVP and UKBB, that were generated in this study.</p>
<ol>
<li>Phe_244&nbsp; &nbsp; &nbsp; &nbsp; Hypothyroidism</li>
<li>Phe_244_4 &nbsp; &nbsp;Hypothyroidism NOS</li>
<li>Phe_250&nbsp; &nbsp; &nbsp; &nbsp; Diabetes mellitus</li>
<li>Phe_250_2 &nbsp; &nbsp;Type 2 diabetes</li>
<li>Phe_278&nbsp; &nbsp; &nbsp; &nbsp; Overweight, obesity and other hyperalimentation</li>
<li>Phe_278_

,CHR,BP,SNP,A1,A2,FREQ,BETA,SE,P,N
0,3,104998275,rs112634005,A,C,0.005466,0.004937,0.010351,0.63360,858458
1,11,134897743,rs7929618,C,G,0.129500,-0.001357,0.002273,0.55070,858458
2,13,33786576,rs151222586,A,G,0.996335,-0.022165,0.012629,0.07932,858458
3,20,8411670,rs6039163,T,C,0.001996,0.024862,0.017099,0.14610,858458
4,16,8600861,rs7190157,A,C,0.344100,-0.001049,0.001606,0.51390,858458



SAMPLE-SIZE INFORMATION IN SUMMARY STATISTICS
Minimum variant-level N: 418949
Maximum variant-level N: 858458
Median variant-level N: 858458.0
Unique N values: 3

DISCOVERY GWAS METADATA

Trait:
Type 2 diabetes mellitus

Study:
Type 2 diabetes mellitus PheCode 250.2 GWAS meta-analysis of Million Veteran Program (MVP) and UK Biobank (UKBB) summary statistics

Population:
MVP participants of African American, European, Hispanic, and East Asian ancestry, combined with UK Biobank participants of European descent

MVP cases:
230291

MVP controls:
382656

MVP total participants:
612947

UKBB total participants:
420531

Total participants:
MVP and UKBB component sample sizes reported separately; the summary-statistics N varies by variant

Genome build:
GRCh37 / hg19

Association method:
SAIGE mixed-model GWAS for MVP, adjusting for age, sex, and the first 10 principal components; MVP summary statistics were meta-analyzed with UK Biobank summary statistics

Source:
Zenodo, DOI: 10.5281/zenodo

## 10. Quality control of the discovery GWAS

Quality control is performed to remove variants that cannot be reliably used for downstream PRS construction.

The QC procedure evaluates:

- missing required fields;
- invalid allele representations;
- invalid allele frequencies;
- invalid effect estimates;
- invalid standard errors;
- invalid P-values; and
- duplicate variant identifiers.

A structured QC report is saved so that every filtering decision can be documented.

In [10]:
# T2D GWAS SUMMARY STATISTICS — QUALITY CONTROL
# Dataset: PheCode 250.2

import time
import shutil

# 1. PATHS

# Raw GWAS summary statistics
GWAS_FILE = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/GWAS/phecode_250_2.tsv.gz"
)

# QC directory
GWAS_QC_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/results/gwas_qc/"
)

GWAS_QC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Final QC outputs on Google Drive
QC_OUTPUT = (
    GWAS_QC_DIR /
    "T2D_PheCode250_2_QC.tsv.gz"
)

QC_REPORT = (
    GWAS_QC_DIR /
    "T2D_PheCode250_2_QC_report.json"
)

# Local temporary files
# Processing locally first avoids repeatedly writing to Google Drive
LOCAL_QC_OUTPUT = Path(
    "/content/T2D_PheCode250_2_QC.tsv.gz"
)

LOCAL_QC_REPORT = Path(
    "/content/T2D_PheCode250_2_QC_report.json"
)

# 2. CHECK INPUT

if not GWAS_FILE.exists():
    raise FileNotFoundError(
        f"GWAS file not found:\n{GWAS_FILE}"
    )

print("T2D GWAS SUMMARY-STATISTICS QC")

print(f"Input file:")
print(GWAS_FILE)

print(
    f"\nInput size: "
    f"{GWAS_FILE.stat().st_size / (1024**3):.2f} GB"
)

# 3. INPUT COLUMNS

usecols = [
    "CHR",
    "BP",
    "SNP",
    "A1",
    "A2",
    "FREQ",
    "BETA",
    "SE",
    "P",
    "N"
]

valid_alleles = {
    "A",
    "C",
    "G",
    "T"
}


# 4. QC STATISTICS

stats = {

    "input_rows": 0,
    "qc_passed_rows": 0,
    "removed_missing": 0,
    "removed_invalid_snp": 0,
    "removed_invalid_alleles": 0,
    "removed_invalid_chr": 0,
    "removed_invalid_bp": 0,
    "removed_invalid_freq": 0,
    "removed_invalid_beta": 0,
    "removed_invalid_se": 0,
    "removed_invalid_p": 0,
    "removed_invalid_n": 0,
    "removed_duplicate_snp": 0,
    "chunks_completed": 0,
}


# 5. INITIALIZE QC

seen_snps = set()
start_time = time.time()
first_write = True

CHUNK_SIZE = 100_000


print("STARTING GWAS QC")

print(f"Chunk size: {CHUNK_SIZE:,}")
print(f"Local QC output: {LOCAL_QC_OUTPUT}")
print()


# 8. READ GWAS IN CHUNKS

reader = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    compression="gzip",
    usecols=usecols,
    chunksize=CHUNK_SIZE,
    low_memory=False,
)


# 9. PROCESS EACH CHUNK

for chunk_number, chunk in enumerate(
    reader,
    start=1
):

    # Count original rows
    stats["input_rows"] += len(chunk)

    # Missing values

    before = len(chunk)

    chunk = chunk.dropna(
        subset=usecols
    )

    stats["removed_missing"] += (
        before - len(chunk)
    )

    # SNP IDENTIFIERS

    chunk["SNP"] = (
        chunk["SNP"]
        .astype(str)
        .str.strip()
    )

    snp_mask = (
        chunk["SNP"].notna()
        & (chunk["SNP"] != "")
        & (
            chunk["SNP"].str.lower()
            != "nan"
        )
    )

    stats["removed_invalid_snp"] += (
        (~snp_mask).sum()
    )

    chunk = chunk[snp_mask]

    # ALLELES

    chunk["A1"] = (
        chunk["A1"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    chunk["A2"] = (
        chunk["A2"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    allele_mask = (
        chunk["A1"].isin(valid_alleles)
        & chunk["A2"].isin(valid_alleles)
        & (
            chunk["A1"]
            != chunk["A2"]
        )
    )

    stats["removed_invalid_alleles"] += (
        (~allele_mask).sum()
    )

    chunk = chunk[allele_mask]


    # NUMERIC CONVERSION

    numeric_columns = [
        "CHR",
        "BP",
        "FREQ",
        "BETA",
        "SE",
        "P",
        "N"
    ]

    for col in numeric_columns:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

    # CHROMOSOME

    chr_mask = (
        np.isfinite(chunk["CHR"])
        & (chunk["CHR"] >= 1)
        & (chunk["CHR"] <= 22)
        & (
            chunk["CHR"]
            == np.floor(chunk["CHR"])
        )
    )

    stats["removed_invalid_chr"] += (
        (~chr_mask).sum()
    )

    chunk = chunk[chr_mask]

    # BASE-PAIR POSITION

    bp_mask = (

        np.isfinite(chunk["BP"])
        & (chunk["BP"] > 0)
        & (
            chunk["BP"]
            == np.floor(chunk["BP"])
        )
    )

    stats["removed_invalid_bp"] += (
        (~bp_mask).sum()
    )

    chunk = chunk[bp_mask]

    # EFFECT ALLELE FREQUENCY

    freq_mask = (

        np.isfinite(chunk["FREQ"])
        & (chunk["FREQ"] >= 0)
        & (chunk["FREQ"] <= 1)
    )

    stats["removed_invalid_freq"] += (
        (~freq_mask).sum()
    )

    chunk = chunk[freq_mask]

    # BETA

    beta_mask = np.isfinite(
        chunk["BETA"]
    )

    stats["removed_invalid_beta"] += (
        (~beta_mask).sum()
    )

    chunk = chunk[beta_mask]

    # STANDARD ERROR

    se_mask = (
        np.isfinite(chunk["SE"])
        & (chunk["SE"] > 0)
    )

    stats["removed_invalid_se"] += (
        (~se_mask).sum()
    )
    chunk = chunk[se_mask]

    # P-VALUE

    p_mask = (
        np.isfinite(chunk["P"])
        & (chunk["P"] > 0)
        & (chunk["P"] <= 1)
    )

    stats["removed_invalid_p"] += (
        (~p_mask).sum()
    )

    chunk = chunk[p_mask]

    # SAMPLE SIZE

    n_mask = (
        np.isfinite(chunk["N"])
        & (chunk["N"] > 0)
    )

    stats["removed_invalid_n"] += (
        (~n_mask).sum()
    )

    chunk = chunk[n_mask]

    # GLOBAL DUPLICATE SNP REMOVAL

    duplicate_mask = (
        chunk["SNP"].isin(seen_snps)
    )

    stats["removed_duplicate_snp"] += (
        duplicate_mask.sum()
    )

    chunk = chunk[
        ~duplicate_mask
    ]

    seen_snps.update(
        chunk["SNP"].tolist()
    )

    # WRITE QC-PASSED CHUNK

    if len(chunk) > 0:

        mode = (
            "wt"
            if first_write
            else "at"
        )

        with gzip.open(

            LOCAL_QC_OUTPUT,
            mode,
            encoding="utf-8",
            newline=""
        ) as gz:

            chunk.to_csv(
                gz,
                sep="\t",
                index=False,
                header=first_write
            )

        first_write = False

        stats["qc_passed_rows"] += (
            len(chunk)
        )

    # PROGRESS

    stats["chunks_completed"] = (
        chunk_number
    )

    elapsed = (
        time.time() - start_time
    ) / 60

    print(

        f"Chunk {chunk_number:03d} | "
        f"Input: "
        f"{stats['input_rows']:,} | "
        f"QC passed: "
        f"{stats['qc_passed_rows']:,} | "
        f"Elapsed: "
        f"{elapsed:.1f} min"
    )

# 10. FINAL QC STATISTICS

stats["runtime_minutes"] = (
    time.time() - start_time
) / 60


# Convert NumPy values to standard Python values

stats_clean = {}

for key, value in stats.items():

    if isinstance(
        value,
        np.integer
    ):

        stats_clean[key] = int(value)

    elif isinstance(
        value,
        np.floating
    ):

        stats_clean[key] = float(value)

    else:

        stats_clean[key] = value

# 11. VERIFY LOCAL GZIP

print("VERIFYING LOCAL QC FILE")


if not LOCAL_QC_OUTPUT.exists():
    raise RuntimeError(
        "Local QC output was not created."
    )

if LOCAL_QC_OUTPUT.stat().st_size == 0:
    raise RuntimeError(
        "Local QC output is empty."
    )

decompressed_bytes = 0

with gzip.open(
    LOCAL_QC_OUTPUT,
    "rb"
) as f:

    while True:

        data = f.read(
            4 * 1024 * 1024
        )

        if not data:
            break

        decompressed_bytes += (
            len(data)
        )


print(
    "Local gzip integrity check passed."
)

print(
    f"Compressed size: "
    f"{LOCAL_QC_OUTPUT.stat().st_size / (1024**3):.2f} GB"
)

print(
    f"Decompressed bytes verified: "
    f"{decompressed_bytes:,}"
)

# 12. COPY VERIFIED FILE TO GOOGLE DRIVE

print(
    "\nCopying verified QC file "
    "to Google Drive..."
)

shutil.copy2(
    LOCAL_QC_OUTPUT,
    QC_OUTPUT
)

print(
    "Copy completed."
)


# ============================================================
# 13. VERIFY GOOGLE DRIVE COPY
# ============================================================

print(
    "\nVerifying Google Drive copy..."
)

drive_decompressed_bytes = 0

with gzip.open(
    QC_OUTPUT,
    "rb"
) as f:

    while True:

        data = f.read(
            4 * 1024 * 1024
        )

        if not data:
            break

        drive_decompressed_bytes += (
            len(data)
        )


if (
    decompressed_bytes
    != drive_decompressed_bytes
):

    raise RuntimeError(

        "Drive copy verification failed: "
        "decompressed byte count differs."
    )


print(
    "Google Drive gzip integrity check passed."
)


# 14. SAVE QC REPORT

stats_clean["output_file"] = str(
    QC_OUTPUT
)

stats_clean[
    "verified_decompressed_bytes"
] = int(
    drive_decompressed_bytes
)


with open(
    QC_REPORT,
    "w"
) as f:

    json.dump(

        stats_clean,

        f,

        indent=2
    )


# 15. FINAL SUMMARY

print("GWAS QC COMPLETE AND VERIFIED")

print(
    json.dumps(
        stats_clean,
        indent=2
    )
)

print(
    f"\nQC output size: "
    f"{QC_OUTPUT.stat().st_size / (1024**3):.2f} GB"
)

print(
    f"\nQC output:"
)

print(QC_OUTPUT)

print(
    f"\nQC report:"
)

print(QC_REPORT)

T2D GWAS SUMMARY-STATISTICS QC
Input file:
/content/drive/MyDrive/PRS_T2D_A-E/GWAS/phecode_250_2.tsv.gz

Input size: 0.77 GB
STARTING GWAS QC
Chunk size: 100,000
Local QC output: /content/T2D_PheCode250_2_QC.tsv.gz

Chunk 001 | Input: 100,000 | QC passed: 98,926 | Elapsed: 0.1 min
Chunk 002 | Input: 200,000 | QC passed: 197,753 | Elapsed: 0.1 min
Chunk 003 | Input: 300,000 | QC passed: 296,545 | Elapsed: 0.2 min
Chunk 004 | Input: 400,000 | QC passed: 395,444 | Elapsed: 0.2 min
Chunk 005 | Input: 500,000 | QC passed: 494,286 | Elapsed: 0.2 min
Chunk 006 | Input: 600,000 | QC passed: 593,180 | Elapsed: 0.3 min
Chunk 007 | Input: 700,000 | QC passed: 691,999 | Elapsed: 0.3 min
Chunk 008 | Input: 800,000 | QC passed: 790,834 | Elapsed: 0.4 min
Chunk 009 | Input: 900,000 | QC passed: 889,707 | Elapsed: 0.4 min
Chunk 010 | Input: 1,000,000 | QC passed: 988,593 | Elapsed: 0.4 min
Chunk 011 | Input: 1,100,000 | QC passed: 1,087,457 | Elapsed: 0.5 min
Chunk 012 | Input: 1,200,000 | QC passed: 

## 11. Harmonization of the QC-passed discovery GWAS

The QC-passed GWAS is converted into a standardized internal representation.

The harmonized dataset uses:

- `SNP` — variant identifier;
- `CHR` — chromosome;
- `BP` — base-pair position;
- `A1` — effect allele;
- `A2` — other allele;
- `EAF` — effect-allele frequency;
- `BETA` — effect estimate;
- `SE` — standard error;
- `P` — association P-value; and
- `N` — sample size where available.

At this stage, target-genotype-specific allele harmonization is not performed because an independent target genotype dataset is not yet available.

In [13]:
# HARMONIZATION — SAFE SINGLE-STREAM GZIP

import gzip
import shutil

# 1. PATHS

QC_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/results/gwas_qc"
)

HARMONIZED_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/results/harmonized"
)

QC_OUTPUT = (
    QC_DIR / "T2D_PheCode250_2_QC.tsv.gz"
)

HARMONIZED_OUTPUT = (
    HARMONIZED_DIR / "T2D_PheCode250_2_harmonized.tsv.gz"
)

LOCAL_HARMONIZED_OUTPUT = Path(
    "/content/T2D PheCode250_2_harmonized.tsv.gz"
)

HARMONIZED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# 2. COLUMNS

input_columns = [
    "CHR", "BP", "SNP",
    "A1", "A2",
    "FREQ", "BETA", "SE", "P", "N"
]

output_columns = [
    "SNP", "CHR", "BP",
    "A1", "A2",
    "EAF", "BETA", "SE", "P", "N"
]

# 3. PROCESS QC FILE

first_write = True
total_rows = 0
start_time = time.time()

print("=" * 60)
print("STARTING GWAS HARMONIZATION")
print("=" * 60)
print(f"Input:  {QC_OUTPUT}")
print(f"Local output: {LOCAL_HARMONIZED_OUTPUT}")
print()

for chunk_number, chunk in enumerate(
    pd.read_csv(
        QC_OUTPUT,
        sep="\t",
        compression="gzip",
        usecols=input_columns,
        chunksize=100_000,
        low_memory=False,
    ),
    start=1
):

    # Rename allele frequency

    chunk = chunk.rename(
        columns={"FREQ": "EAF"}
    )

    # Standardize strings

    chunk["SNP"] = (
        chunk["SNP"]
        .astype(str)
        .str.strip()
    )

    chunk["A1"] = (
        chunk["A1"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    chunk["A2"] = (
        chunk["A2"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # Standardize numeric columns

    for col in [
        "CHR", "BP",
        "EAF", "BETA",
        "SE", "P", "N"
    ]:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )


    # Final column order

    chunk = chunk[output_columns]


    # Write to ONE local gzip stream

    mode = "wt" if first_write else "at"

    with gzip.open(
        LOCAL_HARMONIZED_OUTPUT,
        mode,
        encoding="utf-8",
        newline=""
    ) as gz:

        chunk.to_csv(
            gz,
            sep="\t",
            index=False,
            header=first_write
        )

    first_write = False
    total_rows += len(chunk)

    elapsed = (time.time() - start_time) / 60

    print(
        f"Chunk {chunk_number:03d} | "
        f"Harmonized: {total_rows:,} variants | "
        f"Elapsed: {elapsed:.1f} min"
    )

# 4. CHECK LOCAL OUTPUT

if not LOCAL_HARMONIZED_OUTPUT.exists():
    raise RuntimeError(
        "Local harmonized output was not created."
    )

if LOCAL_HARMONIZED_OUTPUT.stat().st_size == 0:
    raise RuntimeError(
        "Local harmonized output is empty."
    )

print("\nLocal harmonized file created.")

# 5. VERIFY LOCAL GZIP

print("\nVerifying local gzip integrity...")

local_decompressed_bytes = 0

with gzip.open(
    LOCAL_HARMONIZED_OUTPUT,
    "rb"
) as f:

    while True:

        data = f.read(4 * 1024 * 1024)

        if not data:
            break

        local_decompressed_bytes += len(data)

print("✅ Local harmonized gzip is valid.")
print(
    f"Compressed size: "
    f"{LOCAL_HARMONIZED_OUTPUT.stat().st_size / (1024**3):.2f} GB"
)

# 6. COPY VERIFIED FILE TO GOOGLE DRIVE

print("\nCopying verified harmonized file to Drive...")

shutil.copy2(
    LOCAL_HARMONIZED_OUTPUT,
    HARMONIZED_OUTPUT
)

print(" Copy completed.")

# 7. VERIFY DRIVE COPY

print("\nVerifying Drive copy...")

drive_decompressed_bytes = 0

with gzip.open(
    HARMONIZED_OUTPUT,
    "rb"
) as f:

    while True:

        data = f.read(4 * 1024 * 1024)

        if not data:
            break

        drive_decompressed_bytes += len(data)

if local_decompressed_bytes != drive_decompressed_bytes:
    raise RuntimeError(
        "Drive verification failed: "
        "decompressed byte count differs."
    )

print("Drive harmonized gzip is valid.")

# 8. FINAL SUMMARY

runtime_minutes = (
    time.time() - start_time
) / 60

print("\n" + "=" * 60)
print("HARMONIZATION COMPLETE AND VERIFIED")
print("=" * 60)

print(f"Variants: {total_rows:,}")
print(f"Runtime: {runtime_minutes:.2f} minutes")
print(f"Output: {HARMONIZED_OUTPUT}")
print(
    f"Size: "
    f"{HARMONIZED_OUTPUT.stat().st_size / (1024**3):.2f} GB"
)
print(
    f"Verified decompressed bytes: "
    f"{drive_decompressed_bytes:,}"
)

STARTING GWAS HARMONIZATION
Input:  /content/drive/MyDrive/PRS_T2D_A-E/results/gwas_qc/T2D_PheCode250_2_QC.tsv.gz
Local output: /content/T2D PheCode250_2_harmonized.tsv.gz

Chunk 001 | Harmonized: 100,000 variants | Elapsed: 0.0 min
Chunk 002 | Harmonized: 200,000 variants | Elapsed: 0.1 min
Chunk 003 | Harmonized: 300,000 variants | Elapsed: 0.1 min
Chunk 004 | Harmonized: 400,000 variants | Elapsed: 0.1 min
Chunk 005 | Harmonized: 500,000 variants | Elapsed: 0.2 min
Chunk 006 | Harmonized: 600,000 variants | Elapsed: 0.2 min
Chunk 007 | Harmonized: 700,000 variants | Elapsed: 0.2 min
Chunk 008 | Harmonized: 800,000 variants | Elapsed: 0.3 min
Chunk 009 | Harmonized: 900,000 variants | Elapsed: 0.3 min
Chunk 010 | Harmonized: 1,000,000 variants | Elapsed: 0.3 min
Chunk 011 | Harmonized: 1,100,000 variants | Elapsed: 0.4 min
Chunk 012 | Harmonized: 1,200,000 variants | Elapsed: 0.4 min
Chunk 013 | Harmonized: 1,300,000 variants | Elapsed: 0.4 min
Chunk 014 | Harmonized: 1,400,000 varia

## 12. Generate candidate SNP sets at predefined GWAS P-value thresholds

Variants are selected using six predefined association P-value thresholds:

- P ≤ 5 × 10⁻⁸
- P ≤ 1 × 10⁻⁵
- P ≤ 1 × 10⁻⁴
- P ≤ 1 × 10⁻³
- P ≤ 1 × 10⁻²
- P ≤ 5 × 10⁻²

These candidate sets are subsequently subjected to LD clumping using the corresponding threshold-specific GWAS significance level.

In [14]:
# 1. PATHS

CANDIDATE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps"
)

CANDIDATE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# 2. P-VALUE THRESHOLDS

P_THRESHOLDS = [
    5e-8,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
]

# 3. CLEAN PREVIOUS CANDIDATE FILES

for threshold in P_THRESHOLDS:

    output_path = (
        CANDIDATE_DIR
        / f"T2D_PheCode250_2_P{threshold:g}.tsv"
    )

    if output_path.exists():
        output_path.unlink()

# 4. INITIALIZE COUNTS

candidate_counts = {
    threshold: 0
    for threshold in P_THRESHOLDS
}

start_time = time.time()

print("CREATING CANDIDATE SNP SETS")
print(f"Input: {HARMONIZED_OUTPUT}")
print(f"Output directory: {CANDIDATE_DIR}")

# 5. READ HARMONIZED GWAS IN CHUNKS

for chunk_number, chunk in enumerate(
    pd.read_csv(
        HARMONIZED_OUTPUT,
        sep="\t",
        compression="gzip",
        chunksize=100_000,
        low_memory=False,
    ),
    start=1
):

    chunk["P"] = pd.to_numeric(
        chunk["P"],
        errors="coerce"
    )

    # Select SNPs at each P-value threshold

    for threshold in P_THRESHOLDS:

        selected = chunk[
            chunk["P"] <= threshold
        ]

        if selected.empty:
            continue

        output_path = (
            CANDIDATE_DIR
            / f"T2D_PheCode250_2_P{threshold:g}.tsv"
        )

        selected.to_csv(
            output_path,
            sep="\t",
            index=False,
            mode="a",
            header=not output_path.exists(),
        )

        candidate_counts[threshold] += len(selected)

    if chunk_number % 10 == 0:

        elapsed = (
            time.time() - start_time
        ) / 60

        print(
            f"Chunks processed: {chunk_number:,} | "
            f"Elapsed: {elapsed:.1f} min"
        )

# 6. FINAL SUMMARY

print("CANDIDATE SNP SETS CREATED")

for threshold in P_THRESHOLDS:

    output_path = (
        CANDIDATE_DIR
        / f"T2D_PheCode250_2_P{threshold:g}.tsv"
    )

    count = candidate_counts[threshold]

    if count > 0 and output_path.exists():

        print(
            f"P <= {threshold:g}: "
            f"{count:,} SNPs | "
            f"{output_path.name}"
        )

    else:

        print(
            f"P <= {threshold:g}: 0 SNPs"
        )

print(
    f"Runtime: "
    f"{(time.time() - start_time) / 60:.2f} minutes"
)

print(
    f"Output directory: {CANDIDATE_DIR}"
)

CREATING CANDIDATE SNP SETS
Input: /content/drive/MyDrive/PRS_T2D_A-E/results/harmonized/T2D_PheCode250_2_harmonized.tsv.gz
Output directory: /content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps
Chunks processed: 10 | Elapsed: 0.0 min
Chunks processed: 20 | Elapsed: 0.1 min
Chunks processed: 30 | Elapsed: 0.1 min
Chunks processed: 40 | Elapsed: 0.2 min
Chunks processed: 50 | Elapsed: 0.2 min
Chunks processed: 60 | Elapsed: 0.2 min
Chunks processed: 70 | Elapsed: 0.3 min
Chunks processed: 80 | Elapsed: 0.3 min
Chunks processed: 90 | Elapsed: 0.4 min
Chunks processed: 100 | Elapsed: 0.4 min
Chunks processed: 110 | Elapsed: 0.5 min
Chunks processed: 120 | Elapsed: 0.5 min
Chunks processed: 130 | Elapsed: 0.5 min
Chunks processed: 140 | Elapsed: 0.6 min
Chunks processed: 150 | Elapsed: 0.6 min
Chunks processed: 160 | Elapsed: 0.7 min
Chunks processed: 170 | Elapsed: 0.8 min
Chunks processed: 180 | Elapsed: 0.8 min
Chunks processed: 190 | Elapsed: 0.8 min
Chunks processed: 200 | Elapse

In [15]:
#Combine all candidate tsvs

from pathlib import Path
import pandas as pd

# 1. PATHS

CANDIDATE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps"
)

COMBINED_CANDIDATE_FILE = (
    CANDIDATE_DIR
    / "T2D_PheCode250_2_all_candidate_SNPs.tsv.gz"
)

P_THRESHOLDS = [
    5e-8,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
]

# 2. READ AND COMBINE ALL THRESHOLDS

candidate_frames = []

for threshold in P_THRESHOLDS:

    input_path = (
        CANDIDATE_DIR
        / f"T2D_PheCode250_2_P{threshold:g}.tsv"
    )

    if not input_path.exists():
        raise FileNotFoundError(
            f"Candidate file not found: {input_path}"
        )

    df = pd.read_csv(
        input_path,
        sep="\t",
        usecols=["SNP", "P"]
    )

    candidate_frames.append(df)

# 3. COMBINE

combined = pd.concat(
    candidate_frames,
    ignore_index=True
)

# 4. CLEAN AND DEDUPLICATE

combined["SNP"] = (
    combined["SNP"]
    .astype(str)
    .str.strip()
)

combined["P"] = pd.to_numeric(
    combined["P"],
    errors="coerce"
)

combined = (
    combined
    .dropna(subset=["SNP", "P"])
    .sort_values("P")
    .drop_duplicates(subset=["SNP"])
)

# 5. SAVE

combined.to_csv(
    COMBINED_CANDIDATE_FILE,
    sep="\t",
    index=False,
    compression="gzip"
)

print("Combined candidate SNP file created:")
print(COMBINED_CANDIDATE_FILE)
print(f"Unique candidate SNPs: {len(combined):,}")

Combined candidate SNP file created:
/content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps/T2D_PheCode250_2_all_candidate_SNPs.tsv.gz
Unique candidate SNPs: 1,500,187


## 13. Prepare threshold-specific LD-clumping inputs

For each GWAS P-value threshold, a dedicated clumping input file is created.

Each input contains the variant information required by PLINK2 for LD clumping, including:

- SNP identifier;
- chromosome;
- genomic position;
- effect allele;
- other allele; and
- association P-value.

Separate input files allow each threshold to be processed independently and make the clumping stage reproducible.

In [23]:
from pathlib import Path
import pandas as pd

# 1. PATHS

BASE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E"
)

CANDIDATE_DIR = (
    BASE_DIR
    / "results"
    / "candidate_snps"
)

CLUMP_DIR = (
    BASE_DIR
    / "results"
    / "ld_clumping"
)

CLUMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

P_THRESHOLDS = [
    5e-8,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
]

# 2. CREATE CLUMPING INPUTS

print("PREPARING PLINK LD-CLUMPING INPUTS")
print()

for threshold in P_THRESHOLDS:

    candidate_file = (
        CANDIDATE_DIR
        / f"T2D_PheCode250_2_P{threshold:g}.tsv"
    )

    clump_file = (
        CLUMP_DIR
        / f"T2D_PheCode250_2_P{threshold:g}_clump_input.txt"
    )

    print(f"Processing P <= {threshold:g}")
    print(f"Source: {candidate_file}")

    if not candidate_file.exists():
        raise FileNotFoundError(
            f"Candidate file missing:\n{candidate_file}"
        )

    # Read the three columns required for PLINK clumping

    df = pd.read_csv(
        candidate_file,
        sep="\t",
        usecols=["SNP", "A1", "P"],
        dtype={
            "SNP": str,
            "A1": str
        }
    )

    print(
        f"Rows loaded: {len(df):,}"
    )

    # Remove rows missing required fields

    before = len(df)

    df = df.dropna(
        subset=["SNP", "A1", "P"]
    )

    removed = before - len(df)

    if removed > 0:
        print(
            f"Rows removed for missing "
            f"SNP/A1/P: {removed:,}"
        )

    # Standardize SNP and allele strings

    df["SNP"] = (
        df["SNP"]
        .astype(str)
        .str.strip()
    )

    df["A1"] = (
        df["A1"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # Make sure P is numeric

    df["P"] = pd.to_numeric(
        df["P"],
        errors="coerce"
    )

    before = len(df)

    df = df.dropna(
        subset=["P"]
    )

    removed_p = before - len(df)

    if removed_p > 0:
        print(
            f"Rows removed for invalid P: "
            f"{removed_p:,}"
        )

    # Remove duplicate SNP IDs

    before = len(df)

    df = (
        df
        .sort_values("P")
        .drop_duplicates(
            subset=["SNP"],
            keep="first"
        )
    )

    removed_dup = before - len(df)

    if removed_dup > 0:
        print(
            f"Duplicate SNPs removed: "
            f"{removed_dup:,}"
        )

    # Keep exactly the columns needed by PLINK

    df = df[
        ["SNP", "A1", "P"]
    ]

    # Write clumping input

    df.to_csv(
        clump_file,
        sep="\t",
        index=False
    )

    print(
        f"Written: {clump_file}"
    )

    print(
        f"Final rows: {len(df):,}"
    )

    print(
        f"Columns: {list(df.columns)}"
    )

    print()


# 3. VERIFY ALL FILES

print("VERIFYING CLUMPING INPUT FILES")
print()

for threshold in P_THRESHOLDS:

    clump_file = (
        CLUMP_DIR
        / f"T2D_PheCode250_2_P{threshold:g}_clump_input.txt"
    )

    if not clump_file.exists():
        print(
            f"P <= {threshold:g}: MISSING"
        )
        continue

    check = pd.read_csv(
        clump_file,
        sep="\t",
        nrows=5
    )

    print(
        f"P <= {threshold:g}: "
        f"{list(check.columns)} | "
        f"rows readable"
    )

print()
print("PLINK CLUMPING INPUTS READY")
print(
    f"Directory: {CLUMP_DIR}"
)

PREPARING PLINK LD-CLUMPING INPUTS

Processing P <= 5e-08
Source: /content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps/T2D_PheCode250_2_P5e-08.tsv
Rows loaded: 27,292
Written: /content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping/T2D_PheCode250_2_P5e-08_clump_input.txt
Final rows: 27,292
Columns: ['SNP', 'A1', 'P']

Processing P <= 1e-05
Source: /content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps/T2D_PheCode250_2_P1e-05.tsv
Rows loaded: 64,638
Written: /content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping/T2D_PheCode250_2_P1e-05_clump_input.txt
Final rows: 64,638
Columns: ['SNP', 'A1', 'P']

Processing P <= 0.0001
Source: /content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps/T2D_PheCode250_2_P0.0001.tsv
Rows loaded: 104,274
Written: /content/drive/MyDrive/PRS_T2D_A-E/results/ld_clumping/T2D_PheCode250_2_P0.0001_clump_input.txt
Final rows: 104,274
Columns: ['SNP', 'A1', 'P']

Processing P <= 0.001
Source: /content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps/T2D_Phe

##The 1000 Genomes Phase 3 sample metadata were used to identify individuals belonging to the European (EUR) super-population.

The European reference panel includes:

CEU — Utah residents with Northern and Western European ancestry

FIN — Finnish individuals from Finland

GBR — British individuals from England and Scotland

IBS — Iberian individuals in Spain

TSI — Toscani individuals in Italy

The resulting European sample list was saved and reused throughout the pipeline.

The 1000 Genomes Phase 3 genotype files (all_phase3.pgen.zst and all_phase3.pvar.zst) were downloaded manually from the PLINK 2.0 Resources page because their large file sizes caused automated downloads through Google Colab to fail. The sample metadata file (all_phase3.psam) was already available and was reused to identify the European samples.

The PLINK 2.0 Resources page is available at:
https://www.cog-genomics.org/plink/2.0/resources

In [20]:
import pandas as pd
from pathlib import Path

# 14. LOAD AND INSPECT EXISTING 1000 GENOMES PHASE 3 SAMPLE METADATA

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/reference"
)

PSAM_FILE = (
    REFERENCE_DIR / "all_phase3.psam"
)

EUR_SAMPLE_FILE = (
    REFERENCE_DIR / "EUR_samples.tsv"
)

# Check that the existing PSAM file is available
if not PSAM_FILE.exists():
    raise FileNotFoundError(
        f"Existing PSAM file not found:\n{PSAM_FILE}"
    )

print("Using existing 1000 Genomes Phase 3 sample file:")
print(PSAM_FILE)

print(
    f"Size: "
    f"{PSAM_FILE.stat().st_size / 1024:.1f} KB"
)

# Load sample metadata
psam = pd.read_csv(
    PSAM_FILE,
    sep="\t"
)

print("\nShape:", psam.shape)
print("Columns:", psam.columns.tolist())

display(psam.head())

# Identify 1000 Genomes European-superpopulation samples
eur_psam = psam[
    psam["SuperPop"] == "EUR"
].copy()

print(
    "\nTotal 1000 Genomes samples:",
    len(psam)
)

print(
    "European (EUR) samples:",
    len(eur_psam)
)

print("\nEuropean populations:")

print(
    eur_psam["Population"]
    .value_counts()
    .sort_index()
)

print("\nFirst 10 EUR samples:")

display(
    eur_psam.head(10)
)

# Save European sample metadata
eur_psam.to_csv(
    EUR_SAMPLE_FILE,
    sep="\t",
    index=False
)

print(
    f"\nEuropean sample metadata saved to:\n"
    f"{EUR_SAMPLE_FILE}"
)

Using existing 1000 Genomes Phase 3 sample file:
/content/drive/MyDrive/PRS_T2D_A-E/reference/all_phase3.psam
Size: 53.9 KB

Shape: (2504, 6)
Columns: ['#IID', 'PAT', 'MAT', 'SEX', 'SuperPop', 'Population']


,#IID,PAT,MAT,SEX,SuperPop,Population
0,HG00096,0,0,1.0,EUR,GBR
1,HG00097,0,0,2.0,EUR,GBR
2,HG00099,0,0,2.0,EUR,GBR
3,HG00100,0,0,2.0,EUR,GBR
4,HG00101,0,0,1.0,EUR,GBR



Total 1000 Genomes samples: 2504
European (EUR) samples: 503

European populations:
Population
CEU     99
FIN     99
GBR     91
IBS    107
TSI    107
Name: count, dtype: int64

First 10 EUR samples:


,#IID,PAT,MAT,SEX,SuperPop,Population
0,HG00096,0,0,1.0,EUR,GBR
1,HG00097,0,0,2.0,EUR,GBR
2,HG00099,0,0,2.0,EUR,GBR
3,HG00100,0,0,2.0,EUR,GBR
4,HG00101,0,0,1.0,EUR,GBR
5,HG00102,0,0,2.0,EUR,GBR
6,HG00103,0,0,1.0,EUR,GBR
7,HG00105,0,0,1.0,EUR,GBR
8,HG00106,0,0,2.0,EUR,GBR
9,HG00107,0,0,1.0,EUR,GBR



European sample metadata saved to:
/content/drive/MyDrive/PRS_T2D_A-E/reference/EUR_samples.tsv


## Define the European 1000 Genomes reference sample
Select individuals belonging to the EUR superpopulation in 1000 Genomes Phase 3 and record the population composition.
The resulting keep file defines the European-ancestry LD reference called the EUR keep file.

In [24]:
# Create PLINK keep file for EUR samples

EUR_KEEP = (
    REFERENCE_DIR
    / "1000G_EUR.keep"
)

eur_psam[
    ["#IID"]
].assign(
    FID=0
)[
    ["FID", "#IID"]
].to_csv(
    EUR_KEEP,
    sep="\t",
    index=False,
    header=False
)

print("EUR keep file created:")
print(EUR_KEEP)

print(
    "\nNumber of samples in keep file:",
    len(eur_psam)
)

EUR keep file created:
/content/drive/MyDrive/PRS_T2D_A-E/reference/1000G_EUR.keep

Number of samples in keep file: 503


##15. Prepare the 1000 Genomes Phase 3 European LD reference
The 1000 Genomes Phase 3 reference genotype dataset is used as the LD reference panel. The required persistent files are stored in the project reference directory:
- all_phase3.pgen.zst
- all_phase3.pvar.zst
- all_phase3.psam

The European sample list is stored as:
- 1000G_EUR.keep

In [25]:
from pathlib import Path

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/reference"
)

for filename in [
    "all_phase3.pgen.zst",
    "all_phase3.pvar.zst",
    "all_phase3.psam",
    "1000G_EUR.keep",
]:

    path = REFERENCE_DIR / filename

    print(f"\n{filename}")
    print("Exists:", path.exists())

    if path.exists():
        print(
            "Size:",
            round(
                path.stat().st_size / (1024**3),
                2
            ),
            "GB"
        )


all_phase3.pgen.zst
Exists: True
Size: 2.25 GB

all_phase3.pvar.zst
Exists: True
Size: 0.6 GB

all_phase3.psam
Exists: True
Size: 0.0 GB

1000G_EUR.keep
Exists: True
Size: 0.0 GB


In [26]:
from pathlib import Path
import subprocess
import shutil

PLINK = "/content/plink2/plink2"

DRIVE_REF = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/reference"
)

LOCAL_REF = Path(
    "/content/1000G_EUR_reference"
)

LOCAL_REF.mkdir(
    parents=True,
    exist_ok=True
)

PGEN_ZST = DRIVE_REF / "all_phase3.pgen.zst"
PVAR_ZST = DRIVE_REF / "all_phase3.pvar.zst"
PSAM = DRIVE_REF / "all_phase3.psam"
EUR_KEEP = DRIVE_REF / "1000G_EUR.keep"

# Check persistent files

for path in [
    PGEN_ZST,
    PVAR_ZST,
    PSAM,
    EUR_KEEP,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing persistent reference file: {path}"
        )

print("Persistent 1000G EUR reference files found.")

# Copy PVAR, PSAM and EUR keep file locally

for src in [
    PVAR_ZST,
    PSAM,
    EUR_KEEP,
]:

    dst = LOCAL_REF / src.name

    if not dst.exists():
        print(f"Copying {src.name}...")
        shutil.copy2(src, dst)

# Decompress PGEN

LOCAL_PGEN = LOCAL_REF / "all_phase3.pgen"

if LOCAL_PGEN.exists():
    print("Local PGEN already exists.")

else:

    print("Decompressing all_phase3.pgen.zst...")
    print("This may take a while.")

    cmd = [
        PLINK,
        "--zst-decompress",
        str(PGEN_ZST),
        str(LOCAL_PGEN),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:

        print(result.stdout)
        print(result.stderr)

        raise RuntimeError(
            "PGEN decompression failed."
        )

    print("PGEN decompression completed.")

# Verify files

required_local = [
    LOCAL_REF / "all_phase3.pgen",
    LOCAL_REF / "all_phase3.pvar.zst",
    LOCAL_REF / "all_phase3.psam",
    LOCAL_REF / "1000G_EUR.keep",
]

for path in required_local:

    if not path.exists():
        raise FileNotFoundError(
            f"Expected local reference file missing: {path}"
        )

print("1000G EUR REFERENCE RESTORED")

for path in required_local:

    size_gb = path.stat().st_size / (1024**3)

    print(
        f"{path.name}: {size_gb:.2f} GB"
    )

print("\nReady for EUR LD reference extraction.")

Persistent 1000G EUR reference files found.
Copying all_phase3.pvar.zst...
Copying all_phase3.psam...
Copying 1000G_EUR.keep...
Decompressing all_phase3.pgen.zst...
This may take a while.
PGEN decompression completed.
1000G EUR REFERENCE RESTORED
all_phase3.pgen: 6.24 GB
all_phase3.pvar.zst: 0.60 GB
all_phase3.psam: 0.00 GB
1000G_EUR.keep: 0.00 GB

Ready for EUR LD reference extraction.


##16. Construct the European-only LD reference
The complete 1000 Genomes Phase 3 reference is restricted to the predefined European-ancestry samples. This produces an European-only reference dataset that is used for LD estimation and clumping. The resulting files are stored persistently in the project reference directory.

In [27]:
from pathlib import Path
import subprocess

PLINK = "/content/plink2/plink2"

LOCAL_REF = Path("/content/1000G_EUR_reference")
EUR_KEEP = LOCAL_REF / "1000G_EUR.keep"
EUR_REF = LOCAL_REF / "EUR_only"

print("CREATING EUR-ONLY 1000 GENOMES REFERENCE")

cmd = [
    PLINK,
    "--pfile", str(LOCAL_REF / "all_phase3"), "vzs",
    "--keep", str(EUR_KEEP),
    "--make-pgen",
    "--out", str(EUR_REF),
]

print("Running PLINK2...")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("EUR-only reference creation failed.")

print("\nEUR-only reference created.")

for suffix in [".pgen", ".pvar", ".psam"]:
    path = Path(str(EUR_REF) + suffix)

    if not path.exists():
        raise FileNotFoundError(f"Missing output: {path}")

    print(
        f"{path.name}: "
        f"{path.stat().st_size / (1024**2):.1f} MB"
    )

print("\nReady for LD clumping.")

CREATING EUR-ONLY 1000 GENOMES REFERENCE
Running PLINK2...
/content/plink2/plink2 --pfile /content/1000G_EUR_reference/all_phase3 vzs --keep /content/1000G_EUR_reference/1000G_EUR.keep --make-pgen --out /content/1000G_EUR_reference/EUR_only
PLINK v2.0.0-b.1-devLM 64-bit Intel (1 Oct 2026)    cog-genomics.org/plink/2.0/
(C) 2005-2026 S Purcell, C Chang, B Demaille      GNU General Public License v3
Logging to /content/1000G_EUR_reference/EUR_only.log.
Options in effect:
  --keep /content/1000G_EUR_reference/1000G_EUR.keep
  --make-pgen
  --out /content/1000G_EUR_reference/EUR_only
  --pfile /content/1000G_EUR_reference/all_phase3 vzs

Start time: Sun Oct  4 21:16:23 2026
12975 MiB RAM detected, ~7521 available; reserving 6487 MiB for main workspace.
Using up to 2 compute threads.
2504 samples (1270 females, 1233 males, 1 ambiguous; 2497 founders) loaded from
/content/1000G_EUR_reference/all_phase3.psam.
84805772 variants loaded from /content/1000G_EUR_reference/all_phase3.pvar.zst.
2 ca

##17. Verify the European-only LD reference
The generated European reference dataset is checked to ensure that the expected PLINK files exist and can be read successfully. The sample count and variant structure are recorded as a computational checkpoint.

In [28]:
from pathlib import Path
import shutil

LOCAL_EUR = Path("/content/1000G_EUR_reference/EUR_only")

DRIVE_EUR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/reference/EUR_only"
)

DRIVE_EUR.parent.mkdir(
    parents=True,
    exist_ok=True
)

for suffix in [".pgen", ".pvar", ".psam"]:
    src = Path(str(LOCAL_EUR) + suffix)
    dst = Path(str(DRIVE_EUR) + suffix)

    if not src.exists():
        raise FileNotFoundError(
            f"Missing local file: {src}"
        )

    print(f"Copying {src.name} → Drive...")
    shutil.copy2(src, dst)

print("EUR-ONLY REFERENCE SAVED TO GOOGLE DRIVE")

for suffix in [".pgen", ".pvar", ".psam"]:
    path = Path(str(DRIVE_EUR) + suffix)

    print(
        f"{path.name}: "
        f"{path.stat().st_size / (1024**3):.2f} GB"
    )

print("\nPersistent copy complete.")

Copying EUR_only.pgen → Drive...
Copying EUR_only.pvar → Drive...
Copying EUR_only.psam → Drive...
EUR-ONLY REFERENCE SAVED TO GOOGLE DRIVE
EUR_only.pgen: 1.27 GB
EUR_only.pvar: 2.16 GB
EUR_only.psam: 0.00 GB

Persistent copy complete.


## 18. Create a candidate SNP lookup set

The union of all GWAS candidate SNPs across the six P-value thresholds is used to create a lookup set.

This lookup set is used to identify which candidate GWAS variants are present in the European 1000 Genomes reference.

Using the union of candidate SNPs avoids repeatedly scanning the reference for the same variants.

In [29]:
from pathlib import Path
import pandas as pd

# 1. PATHS

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/reference"
)

CANDIDATE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps"
)

# European-only reference prefix

EUR_ONLY = REFERENCE_DIR / "EUR_only"

# Combined candidate SNP file
# These candidates come from the AFRICAN GWAS

CANDIDATE_SNP_LIST = (
    CANDIDATE_DIR
    / "T2D_PheCode250_2_all_candidate_SNPs.tsv.gz"
)

# European-only reference PVAR

EUR_PVAR = Path(
    str(EUR_ONLY) + ".pvar"
)

# Output

OUTPUT_FILE = (
    CANDIDATE_DIR
    / "T2D_PheCode250_2_AFR_candidate_SNPs_in_1000G_EUR.txt"
)

print(
    "Candidate SNP list:",
    CANDIDATE_SNP_LIST
)

print(
    "EUR PVAR:",
    EUR_PVAR
)

# 2. CHECK REQUIRED FILES

if not CANDIDATE_SNP_LIST.exists():
    raise FileNotFoundError(
        f"Candidate SNP list not found:\n"
        f"{CANDIDATE_SNP_LIST}"
    )

if not EUR_PVAR.exists():
    raise FileNotFoundError(
        f"European reference PVAR not found:\n"
        f"{EUR_PVAR}"
    )

print("\nAll required input files exist.")

# 3. LOAD AFRICAN GWAS CANDIDATE SNPs

candidate_df = pd.read_csv(
    CANDIDATE_SNP_LIST,
    sep="\t",
    compression="gzip",
    usecols=["SNP"]
)

candidate_snps = set(
    candidate_df["SNP"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(
    f"AFR GWAS candidate SNPs: "
    f"{len(candidate_snps):,}"
)

Candidate SNP list: /content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps/T2D_PheCode250_2_all_candidate_SNPs.tsv.gz
EUR PVAR: /content/drive/MyDrive/PRS_T2D_A-E/reference/EUR_only.pvar

All required input files exist.
AFR GWAS candidate SNPs: 1,500,187


## 19. Match GWAS candidate SNPs to the European LD reference PVAR

The candidate GWAS variants are matched against the European reference PVAR.

The PVAR is scanned sequentially rather than loading the entire reference variant table into memory. This is necessary because the 1000 Genomes reference contains tens of millions of variants and the available computational environment has limited RAM.

The output identifies candidate SNPs that are present in the European reference panel.

In [30]:
# MATCH AFRICAN GWAS CANDIDATE SNPs TO EUROPEAN LD REFERENCE PVAR

matched_snps = []

with open(EUR_PVAR, "r", encoding="utf-8") as f:

    # Skip PVAR metadata lines and locate header
    while True:
        line = f.readline()

        if not line:
            raise ValueError(
                "Reached end of PVAR before finding header."
            )

        if not line.startswith("##"):
            header = line.rstrip("\n").split("\t")
            break

    print("PVAR header:")
    print(header)

    # Locate variant ID column
    if "ID" in header:
        id_idx = header.index("ID")
    elif "#ID" in header:
        id_idx = header.index("#ID")
    else:
        raise ValueError(
            f"Could not find variant ID column in PVAR header: {header}"
        )

    print(f"\nVariant ID column index: {id_idx}")
    print(f"Variant ID column name: {header[id_idx]}")

    # Scan the PVAR sequentially without loading it into memory
    for line_number, line in enumerate(f, start=2):

        fields = line.rstrip("\n").split("\t")

        if len(fields) <= id_idx:
            continue

        snp = fields[id_idx]

        if snp in candidate_snps:
            matched_snps.append(snp)

        if line_number % 5_000_000 == 0:
            print(
                f"Scanned {line_number:,} PVAR rows | "
                f"Matched {len(matched_snps):,}"
            )

print("\nPVAR scan complete.")

# Remove duplicate matches

matched_snps = sorted(set(matched_snps))

print(
    f"Candidate SNPs found in EUR reference: "
    f"{len(matched_snps):,}"
)

# Save matched candidate SNP list

MATCHED_SNP_LIST = (
    CANDIDATE_DIR
    / "T2D_PheCode250_2_AFR_candidate_SNPs_in_1000G_EUR.txt"
)

pd.Series(
    matched_snps,
    dtype="string"
).to_csv(
    MATCHED_SNP_LIST,
    index=False,
    header=False
)

print("\nMatched SNP list saved:")
print(MATCHED_SNP_LIST)

print(
    f"Unique matched SNPs: "
    f"{len(matched_snps):,}"
)

PVAR header:
['#CHROM', 'POS', 'ID', 'REF', 'ALT']

Variant ID column index: 2
Variant ID column name: ID
Scanned 5,000,000 PVAR rows | Matched 86,193
Scanned 10,000,000 PVAR rows | Matched 172,548
Scanned 15,000,000 PVAR rows | Matched 257,932
Scanned 20,000,000 PVAR rows | Matched 350,644
Scanned 25,000,000 PVAR rows | Matched 432,508
Scanned 30,000,000 PVAR rows | Matched 524,089
Scanned 35,000,000 PVAR rows | Matched 620,920
Scanned 40,000,000 PVAR rows | Matched 711,724
Scanned 45,000,000 PVAR rows | Matched 803,667
Scanned 50,000,000 PVAR rows | Matched 888,522
Scanned 55,000,000 PVAR rows | Matched 983,712
Scanned 60,000,000 PVAR rows | Matched 1,072,753
Scanned 65,000,000 PVAR rows | Matched 1,157,487
Scanned 70,000,000 PVAR rows | Matched 1,240,290
Scanned 75,000,000 PVAR rows | Matched 1,328,548
Scanned 80,000,000 PVAR rows | Matched 1,401,960

PVAR scan complete.
Candidate SNPs found in EUR reference: 1,424,010

Matched SNP list saved:
/content/drive/MyDrive/PRS_T2D_A-E/resu

## 20. Create chromosome-specific candidate SNP extraction lists

The matched candidate SNPs are separated by chromosome.

A separate extraction list is created for each chromosome so that chromosome-specific European reference datasets can be constructed.

This chromosome-wise strategy reduces memory requirements during downstream PLINK processing.

In [31]:
from pathlib import Path
import pandas as pd

# 1. PATHS

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/reference"
)

CANDIDATE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/results/candidate_snps"
)

MATCHED_SNP_LIST = (
    CANDIDATE_DIR
    / "T2D_PheCode250_2_AFR_candidate_SNPs_in_1000G_EUR.txt"
)

CHUNK_DIR = (
    REFERENCE_DIR
    / "AE_candidate_chunks"
)

CHUNK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not MATCHED_SNP_LIST.exists():
    raise FileNotFoundError(
        f"Matched SNP list not found:\n{MATCHED_SNP_LIST}"
    )

# 2. READ MATCHED SNPs

matched = pd.read_csv(
    MATCHED_SNP_LIST,
    header=None,
    names=["SNP"],
    dtype="string"
)

matched = matched.dropna()

matched["SNP"] = (
    matched["SNP"]
    .str.strip()
)

matched = matched.drop_duplicates()

print(
    f"Matched SNPs: {len(matched):,}"
)

# 3. SCAN EUR PVAR FOR CHROMOSOME INFORMATION

matched_set = set(
    matched["SNP"]
)

chromosome_snps = {}

EUR_PVAR = Path(
    str(REFERENCE_DIR / "EUR_only") + ".pvar"
)

if not EUR_PVAR.exists():
    raise FileNotFoundError(
        f"European PVAR not found:\n{EUR_PVAR}"
    )

print("\nScanning EUR PVAR...")

with open(
    EUR_PVAR,
    "r",
    encoding="utf-8"
) as f:

    # Skip ## metadata lines

    while True:

        line = f.readline()

        if not line:
            raise ValueError(
                "Could not find PVAR header."
            )

        if not line.startswith("##"):
            header = (
                line
                .rstrip("\n")
                .split("\t")
            )
            break

    if "#CHROM" not in header:
        raise ValueError(
            f"#CHROM column not found in PVAR header: {header}"
        )

    if "ID" not in header:
        raise ValueError(
            f"ID column not found in PVAR header: {header}"
        )

    chrom_idx = header.index("#CHROM")
    id_idx = header.index("ID")

    print(
        f"Chromosome column index: {chrom_idx}"
    )

    print(
        f"Variant ID column index: {id_idx}"
    )

    # Scan variants without loading
    # the entire PVAR into memory

    for line_number, line in enumerate(
        f,
        start=1
    ):

        fields = (
            line
            .rstrip("\n")
            .split("\t")
        )

        if len(fields) <= max(
            chrom_idx,
            id_idx
        ):
            continue

        chrom = fields[chrom_idx]
        snp = fields[id_idx]

        if snp in matched_set:

            chromosome_snps.setdefault(
                chrom,
                []
            ).append(snp)

        if line_number % 5_000_000 == 0:

            print(
                f"Scanned "
                f"{line_number:,} variants"
            )

print("\nPVAR scan complete.")

# 4. SAVE ONE EXTRACTION LIST PER CHROMOSOME

total_written = 0

for chrom, snps in sorted(
    chromosome_snps.items(),
    key=lambda x: str(x[0])
):

    output = (
        CHUNK_DIR
        / f"chr{chrom}_extract.txt"
    )

    unique_snps = sorted(
        set(snps)
    )

    pd.Series(
        unique_snps,
        dtype="string"
    ).to_csv(
        output,
        index=False,
        header=False
    )

    total_written += len(unique_snps)

    print(
        f"chr{chrom}: "
        f"{len(unique_snps):,} SNPs -> "
        f"{output.name}"
    )

print(
    "\nChromosome extraction lists created."
)

print(
    f"Total unique SNPs written: "
    f"{total_written:,}"
)

print(
    f"Directory: {CHUNK_DIR}"
)

Matched SNPs: 1,424,010

Scanning EUR PVAR...
Chromosome column index: 0
Variant ID column index: 2
Scanned 5,000,000 variants
Scanned 10,000,000 variants
Scanned 15,000,000 variants
Scanned 20,000,000 variants
Scanned 25,000,000 variants
Scanned 30,000,000 variants
Scanned 35,000,000 variants
Scanned 40,000,000 variants
Scanned 45,000,000 variants
Scanned 50,000,000 variants
Scanned 55,000,000 variants
Scanned 60,000,000 variants
Scanned 65,000,000 variants
Scanned 70,000,000 variants
Scanned 75,000,000 variants
Scanned 80,000,000 variants

PVAR scan complete.
chr1: 112,342 SNPs -> chr1_extract.txt
chr10: 64,752 SNPs -> chr10_extract.txt
chr11: 78,920 SNPs -> chr11_extract.txt
chr12: 66,953 SNPs -> chr12_extract.txt
chr13: 50,236 SNPs -> chr13_extract.txt
chr14: 43,881 SNPs -> chr14_extract.txt
chr15: 42,614 SNPs -> chr15_extract.txt
chr16: 42,985 SNPs -> chr16_extract.txt
chr17: 44,610 SNPs -> chr17_extract.txt
chr18: 33,869 SNPs -> chr18_extract.txt
chr19: 27,418 SNPs -> chr19_extra

## 21. Construct chromosome-specific candidate-restricted European LD references

Each chromosome-specific extraction list is applied to the European 1000 Genomes reference.

This produces candidate-restricted LD reference datasets containing only the variants required for downstream LD clumping.

Processing is performed chromosome by chromosome to reduce memory requirements.

The resulting reference datasets are stored as persistent checkpoint files.

In [32]:
import subprocess
import os

# 1. PATHS

base = "/content/drive/MyDrive/PRS_T2D_A-E/"

plink2 = "/content/plink2/plink2"

# Existing European-only reference

eur_reference = base + "reference/EUR_only"

# Chromosome-specific candidate SNP lists

extract_dir = (
    base +
    "reference/AE_candidate_chunks/"
)

# Persistent chromosome-specific LD references

chunk_dir = (
    base +
    "results/ld_clumping/reference_chunks/"
)

os.makedirs(
    chunk_dir,
    exist_ok=True
)

print(
    "Creating chromosome-specific EUR "
    "LD reference datasets...\n"
)

# 2. PROCESS CHROMOSOMES

for chrom in range(1, 23):

    out_prefix = (
        chunk_dir +
        f"EUR_candidate_chr{chrom}_nodup"
    )

    extract_file = (
        extract_dir +
        f"chr{chrom}_extract.txt"
    )

    # Check whether extract list exists

    if not os.path.exists(extract_file):

        print(
            f"SKIP chr{chrom}: "
            f"extract file not found"
        )

        continue

    # Skip if already successfully created

    required = [
        out_prefix + ".pgen",
        out_prefix + ".pvar",
        out_prefix + ".psam"
    ]

    if all(
        os.path.exists(x)
        for x in required
    ):

        print(
            f"SKIP chr{chrom}: "
            f"reference already exists"
        )

        continue

    # PLINK2 command

    cmd = [
        plink2,
        "--pfile",
        eur_reference,
        "--chr",
        str(chrom),
        "--extract",
        extract_file,
        "--rm-dup",
        "force-first",
        "--make-pgen",
        "--memory",
        "9000",
        "--out",
        out_prefix
    ]

    print(
        f"Processing chromosome {chrom}..."
    )

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:

        print(
            f"OK chr{chrom}"
        )

    else:

        print(
            f"FAILED chr{chrom} "
            f"— exit code {result.returncode}"
        )

        print(
            result.stderr[-1000:]
        )

print(
    "\nChromosome-specific EUR reference "
    "construction finished."
)

Creating chromosome-specific EUR LD reference datasets...

Processing chromosome 1...
OK chr1
Processing chromosome 2...
OK chr2
Processing chromosome 3...
OK chr3
Processing chromosome 4...
OK chr4
Processing chromosome 5...
OK chr5
Processing chromosome 6...
OK chr6
Processing chromosome 7...
OK chr7
Processing chromosome 8...
OK chr8
Processing chromosome 9...
OK chr9
Processing chromosome 10...
OK chr10
Processing chromosome 11...
OK chr11
Processing chromosome 12...
OK chr12
Processing chromosome 13...
OK chr13
Processing chromosome 14...
OK chr14
Processing chromosome 15...
OK chr15
Processing chromosome 16...
OK chr16
Processing chromosome 17...
OK chr17
Processing chromosome 18...
OK chr18
Processing chromosome 19...
OK chr19
Processing chromosome 20...
OK chr20
Processing chromosome 21...
OK chr21
Processing chromosome 22...
OK chr22

Chromosome-specific EUR reference construction finished.


## 22. Verify chromosome-specific European reference datasets

Each chromosome-specific reference dataset is checked for the expected PLINK file components.

This verification ensures that all chromosome-specific references required for LD clumping were successfully generated before computationally expensive clumping begins.

In [33]:
import os

chunk_dir = (
    "/content/drive/MyDrive/PRS_T2D_A-E/"
    "results/ld_clumping/reference_chunks/"
)

print("Chromosome-specific AE reference verification\n")

total_ok = 0

for chrom in range(1, 23):

    prefix = (
        chunk_dir +
        f"EUR_candidate_chr{chrom}_nodup"
    )

    pgen = prefix + ".pgen"
    pvar = prefix + ".pvar"
    psam = prefix + ".psam"

    if (
        os.path.exists(pgen)
        and os.path.exists(pvar)
        and os.path.exists(psam)
    ):

        with open(pvar, "r") as f:

            n_var = sum(
                1
                for line in f
                if not line.startswith("#")
            )

        print(
            f"OK chr{chrom}: "
            f"{n_var:,} variants"
        )

        total_ok += 1

    else:

        missing = []

        if not os.path.exists(pgen):
            missing.append(".pgen")

        if not os.path.exists(pvar):
            missing.append(".pvar")

        if not os.path.exists(psam):
            missing.append(".psam")

        print(
            f"FAILED chr{chrom}: "
            f"missing {missing}"
        )

print(
    f"\n{total_ok}/22 chromosomes complete."
)

Chromosome-specific AE reference verification

OK chr1: 112,342 variants
OK chr2: 121,629 variants
OK chr3: 107,259 variants
OK chr4: 92,767 variants
OK chr5: 94,982 variants
OK chr6: 99,895 variants
OK chr7: 85,248 variants
OK chr8: 85,444 variants
OK chr9: 65,369 variants
OK chr10: 64,752 variants
OK chr11: 78,920 variants
OK chr12: 66,953 variants
OK chr13: 50,236 variants
OK chr14: 43,881 variants
OK chr15: 42,614 variants
OK chr16: 42,985 variants
OK chr17: 44,610 variants
OK chr18: 33,869 variants
OK chr19: 27,418 variants
OK chr20: 30,239 variants
OK chr21: 13,701 variants
OK chr22: 18,897 variants

22/22 chromosomes complete.


## 23. Preliminary chromosome-wise LD clumping

A preliminary chromosome-wise LD clumping run is performed using the full harmonized GWAS and `P1 = 1`.

This stage serves as a computational/reference check of the chromosome-specific clumping infrastructure.

The outputs are stored separately from the threshold-specific clumping results and are not used to construct the final PRS weights.

In [34]:
import subprocess
import os

base = "/content/drive/MyDrive/PRS_T2D_A-E/"
plink2 = "/content/plink2/plink2"

# Chromosome-specific European LD reference datasets

chunk_dir = (
    base +
    "results/ld_clumping/reference_chunks/"
)

# Harmonized AFRICAN GWAS

harm = (
    base +
    "results/harmonized/"
    "T2D_PheCode250_2_harmonized.tsv.gz"
)

# Preliminary clumping output directory

clump_dir = (
    base +
    "results/ld_clumping/normal_clumps/"
)

os.makedirs(
    clump_dir,
    exist_ok=True
)

print(
    "Starting chromosome-wise AFR GWAS / EUR LD clumping...\n"
)

for chrom in range(1, 23):

    reference_prefix = (
        chunk_dir +
        f"EUR_candidate_chr{chrom}_nodup"
    )

    clump_prefix = (
        clump_dir +
        f"clump_chr{chrom}_nodup"
    )

    # Required European reference files

    required = [
        reference_prefix + ".pgen",
        reference_prefix + ".pvar",
        reference_prefix + ".psam"
    ]

    if not all(
        os.path.exists(x)
        for x in required
    ):

        print(
            f"FAILED chr{chrom}: "
            f"reference dataset missing"
        )

        continue

    cmd = [
        plink2,
        "--pfile",
        reference_prefix,
        "--clump",
        harm,
        "--clump-p1",
        "1",
        "--clump-r2",
        "0.1",
        "--clump-kb",
        "250",
        "--memory",
        "9000",
        "--out",
        clump_prefix
    ]

    print(
        f"Running LD clumping for chromosome {chrom}..."
    )

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    clumps_file = (
        clump_prefix +
        ".clumps"
    )

    if (
        result.returncode == 0
        and os.path.exists(clumps_file)
    ):

        size_kb = (
            os.path.getsize(clumps_file)
            / 1024
        )

        print(
            f"OK chr{chrom}: "
            f"{os.path.basename(clumps_file)} "
            f"({size_kb:.1f} KB)"
        )

    else:

        print(
            f"FAILED chr{chrom}: "
            f"PLINK exit code "
            f"{result.returncode}"
        )

        if result.stderr:
            print(
                result.stderr[-1500:]
            )

print(
    "\nAFR GWAS / EUR LD clumping loop finished."
)

Starting chromosome-wise AFR GWAS / EUR LD clumping...

Running LD clumping for chromosome 1...
OK chr1: clump_chr1_nodup.clumps (1659.9 KB)
Running LD clumping for chromosome 2...
OK chr2: clump_chr2_nodup.clumps (1784.3 KB)
Running LD clumping for chromosome 3...
OK chr3: clump_chr3_nodup.clumps (1464.5 KB)
Running LD clumping for chromosome 4...
OK chr4: clump_chr4_nodup.clumps (1352.6 KB)
Running LD clumping for chromosome 5...
OK chr5: clump_chr5_nodup.clumps (1363.8 KB)
Running LD clumping for chromosome 6...
OK chr6: clump_chr6_nodup.clumps (1435.2 KB)
Running LD clumping for chromosome 7...
OK chr7: clump_chr7_nodup.clumps (1244.9 KB)
Running LD clumping for chromosome 8...
OK chr8: clump_chr8_nodup.clumps (1279.1 KB)
Running LD clumping for chromosome 9...
OK chr9: clump_chr9_nodup.clumps (999.0 KB)
Running LD clumping for chromosome 10...
OK chr10: clump_chr10_nodup.clumps (1029.7 KB)
Running LD clumping for chromosome 11...
OK chr11: clump_chr11_nodup.clumps (1097.7 KB)
Runn

In [35]:
import os

clump_dir = (
    "/content/drive/MyDrive/PRS_T2D_A-E/"
    "results/ld_clumping/normal_clumps/"
)

print("PRELIMINARY EUR CLUMPING CHECK\n")

completed = []

for chrom in range(1, 23):

    prefix = (
        clump_dir +
        f"clump_chr{chrom}_nodup"
    )

    clumps_file = (
        prefix +
        ".clumps"
    )

    if os.path.exists(clumps_file):

        size_kb = (
            os.path.getsize(clumps_file)
            / 1024
        )

        print(
            f"OK chr{chrom}: "
            f"{size_kb:.1f} KB"
        )

        completed.append(chrom)

    else:

        print(
            f"MISSING chr{chrom}"
        )

print(
    f"\nCompleted: "
    f"{len(completed)}/22 chromosomes"
)

if len(completed) == 22:

    print(
        "ALL 22 CHROMOSOMES COMPLETED."
    )

else:

    missing = [
        c
        for c in range(1, 23)
        if c not in completed
    ]

    print(
        "Missing:",
        missing
    )

PRELIMINARY EUR CLUMPING CHECK

OK chr1: 1659.9 KB
OK chr2: 1784.3 KB
OK chr3: 1464.5 KB
OK chr4: 1352.6 KB
OK chr5: 1363.8 KB
OK chr6: 1435.2 KB
OK chr7: 1244.9 KB
OK chr8: 1279.1 KB
OK chr9: 999.0 KB
OK chr10: 1029.7 KB
OK chr11: 1097.7 KB
OK chr12: 1062.5 KB
OK chr13: 720.0 KB
OK chr14: 661.0 KB
OK chr15: 642.4 KB
OK chr16: 659.6 KB
OK chr17: 664.1 KB
OK chr18: 542.4 KB
OK chr19: 435.0 KB
OK chr20: 488.6 KB
OK chr21: 228.3 KB
OK chr22: 264.4 KB

Completed: 22/22 chromosomes
ALL 22 CHROMOSOMES COMPLETED.


## 24. Threshold-specific chromosome-wise LD clumping

LD clumping is performed separately for each of the six GWAS P-value thresholds.

For each threshold:

- the corresponding candidate SNP set is used;
- chromosome-specific European reference data are used;
- the index-variant significance threshold is set to the corresponding GWAS P-value threshold;
- the LD threshold is set to r² = 0.1;
- the physical distance window is set to 250 kb; and
- secondary clumping variants are allowed up to P = 1.0.

Chromosome-wise processing reduces memory requirements and allows failed jobs to be identified and rerun independently.

There are:

6 thresholds × 22 autosomes = 132 autosomal clumping jobs.

If chromosome X is intentionally included, it should be documented and processed separately rather than silently mixing it into the 132-job count.

In [36]:
import os
import subprocess
from pathlib import Path

PLINK = "/content/plink2/plink2"

base = Path(
    "/content/drive/MyDrive/PRS_T2D_A-E/"
)

reference_dir = (
    base /
    "results/ld_clumping/reference_chunks"
)

input_dir = (
    base /
    "results/ld_clumping"
)

output_dir = (
    base /
    "results/ld_clumping/threshold_clumps"
)

thresholds = {
    "P5e-08": 5e-8,
    "P1e-05": 1e-5,
    "P0.0001": 1e-4,
    "P0.001": 1e-3,
    "P0.01": 1e-2,
    "P0.05": 5e-2
}

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

success = []
failed = []
skipped = []

total_jobs = len(thresholds) * 22
completed_jobs = 0

print("T2D PRS AFRICAN GWAS / EUROPEAN LD CLUMPING")
print(f"Total combinations: {total_jobs}")
print("Thresholds: 6")
print("Chromosomes: 22")
print("Memory per PLINK process: 9000 MB")

for threshold_name, threshold in thresholds.items():

    print(
        f"\nTHRESHOLD: {threshold_name} "
        f"(P <= {threshold})"
    )

    # AFR GWAS threshold-specific clumping input

    clump_input = (
        input_dir /
        f"T2D_PheCode250_2_{threshold_name}_clump_input.txt"
    )

    if not clump_input.exists():

        print(
            f"ERROR: Input file not found:\n"
            f"{clump_input}"
        )

        for chrom in range(1, 23):

            failed.append(
                (
                    threshold_name,
                    chrom,
                    "missing_input"
                )
            )

        continue

    threshold_out = (
        output_dir /
        threshold_name
    )

    threshold_out.mkdir(
        parents=True,
        exist_ok=True
    )

    for chrom in range(1, 23):

        reference_prefix = (
            reference_dir /
            f"EUR_candidate_chr{chrom}_nodup"
        )

        output_prefix = (
            threshold_out /
            f"clump_chr{chrom}"
        )

        clumps_file = Path(
            str(output_prefix) +
            ".clumps"
        )

        completed_jobs += 1

        print(
            f"\n[{completed_jobs}/{total_jobs}] "
            f"{threshold_name} | chr{chrom}"
        )

        # CHECK REFERENCE

        required_reference_files = [
            Path(
                str(reference_prefix) +
                ".pgen"
            ),
            Path(
                str(reference_prefix) +
                ".pvar"
            ),
            Path(
                str(reference_prefix) +
                ".psam"
            )
        ]

        missing_reference = [
            str(f)
            for f in required_reference_files
            if not f.exists()
        ]

        if missing_reference:

            print(
                "FAILED: Missing reference file(s)"
            )

            for f in missing_reference:
                print(f"  {f}")

            failed.append(
                (
                    threshold_name,
                    chrom,
                    "missing_reference"
                )
            )

            continue

        # SKIP EXISTING SUCCESSFUL OUTPUT

        if clumps_file.exists():

            print(
                "SKIPPED: .clumps already exists"
            )

            print(
                f"Existing file: {clumps_file}"
            )

            skipped.append(
                (
                    threshold_name,
                    chrom
                )
            )

            continue

        # PLINK COMMAND

        cmd = [
            PLINK,

            "--pfile",
            str(reference_prefix),

            "--clump",
            str(clump_input),

            "--clump-p1",
            str(threshold),

            "--clump-p2",
            "1",

            "--clump-r2",
            "0.1",

            "--clump-kb",
            "250",

            "--memory",
            "9000",

            "--out",
            str(output_prefix)
        ]

        print("Running PLINK...")

        try:

            result = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=1800
            )

        except subprocess.TimeoutExpired:

            print(
                "FAILED: PLINK timed out after "
                "30 minutes."
            )

            failed.append(
                (
                    threshold_name,
                    chrom,
                    "timeout"
                )
            )

            continue

        except Exception as e:

            print(
                "FAILED: Python could not run PLINK."
            )

            print(repr(e))

            failed.append(
                (
                    threshold_name,
                    chrom,
                    "python_error"
                )
            )

            continue

        # DETERMINE SUCCESS

        if clumps_file.exists():

            print(
                f"SUCCESS: {threshold_name} "
                f"chr{chrom}"
            )

            success.append(
                (
                    threshold_name,
                    chrom
                )
            )

        else:

            print(
                f"FAILED: {threshold_name} "
                f"chr{chrom}"
            )

            print(
                f"PLINK return code: "
                f"{result.returncode}"
            )

            if result.stderr:

                print(
                    "\nPLINK STDERR:"
                )

                print(
                    result.stderr[-2500:]
                )

            if result.stdout:

                print(
                    "\nPLINK STDOUT:"
                )

                print(
                    result.stdout[-2500:]
                )

            failed.append(
                (
                    threshold_name,
                    chrom,
                    f"return_code_{result.returncode}"
                )
            )


# FINAL SUMMARY

print("\n")
print("CLUMPING SUMMARY")

print(
    f"Successful new runs:       "
    f"{len(success)}"
)

print(
    f"Skipped existing outputs:  "
    f"{len(skipped)}"
)

print(
    f"Failed runs:               "
    f"{len(failed)}"
)

print(
    f"Accounted for:             "
    f"{len(success) + len(skipped) + len(failed)}"
    f"/{total_jobs}"
)

if skipped:

    print("\nSkipped:")

    for threshold_name, chrom in skipped:

        print(
            f"  {threshold_name} - chr{chrom}"
        )

if failed:

    print("\nFAILED:")

    for threshold_name, chrom, reason in failed:

        print(
            f"  {threshold_name} - chr{chrom} "
            f"[{reason}]"
        )

else:

    print(
        "\nALL 132 CLUMPING COMBINATIONS "
        "ACCOUNTED FOR."
    )

print("\nDone.")

T2D PRS AFRICAN GWAS / EUROPEAN LD CLUMPING
Total combinations: 132
Thresholds: 6
Chromosomes: 22
Memory per PLINK process: 9000 MB

THRESHOLD: P5e-08 (P <= 5e-08)

[1/132] P5e-08 | chr1
Running PLINK...
SUCCESS: P5e-08 chr1

[2/132] P5e-08 | chr2
Running PLINK...
SUCCESS: P5e-08 chr2

[3/132] P5e-08 | chr3
Running PLINK...
SUCCESS: P5e-08 chr3

[4/132] P5e-08 | chr4
Running PLINK...
SUCCESS: P5e-08 chr4

[5/132] P5e-08 | chr5
Running PLINK...
SUCCESS: P5e-08 chr5

[6/132] P5e-08 | chr6
Running PLINK...
SUCCESS: P5e-08 chr6

[7/132] P5e-08 | chr7
Running PLINK...
SUCCESS: P5e-08 chr7

[8/132] P5e-08 | chr8
Running PLINK...
SUCCESS: P5e-08 chr8

[9/132] P5e-08 | chr9
Running PLINK...
SUCCESS: P5e-08 chr9

[10/132] P5e-08 | chr10
Running PLINK...
SUCCESS: P5e-08 chr10

[11/132] P5e-08 | chr11
Running PLINK...
SUCCESS: P5e-08 chr11

[12/132] P5e-08 | chr12
Running PLINK...
SUCCESS: P5e-08 chr12

[13/132] P5e-08 | chr13
Running PLINK...
SUCCESS: P5e-08 chr13

[14/132] P5e-08 | chr14
Runnin

## 25. Verify threshold-specific LD clumping

The outputs from all threshold-specific chromosome-wise clumping jobs are checked.

The verification confirms:

- the expected number of jobs completed;
- expected `.clumps` files exist;
- files are non-empty; and
- each threshold has results across the expected chromosomes.

A summary table is generated to document the computational completion status.

In [37]:
import os
import pandas as pd

base = "/content/drive/MyDrive/PRS_T2D_A-E/"
clump_base = base + "results/ld_clumping/threshold_clumps/"

thresholds = [
    "P5e-08",
    "P1e-05",
    "P0.0001",
    "P0.001",
    "P0.01",
    "P0.05"
]

print("THRESHOLD-SPECIFIC CLUMPING AUDIT")
print("=" * 70)

summary = []

for threshold in thresholds:

    successful = 0
    empty = 0
    missing = 0
    total_snps = 0

    for chrom in range(1, 23):

        file = (
            clump_base
            + threshold
            + f"/clump_chr{chrom}.clumps"
        )

        if not os.path.exists(file):
            missing += 1
            continue

        if os.path.getsize(file) == 0:
            empty += 1
            continue

        try:
            df = pd.read_csv(
                file,
                sep=r"\s+"
            )

            n = len(df)

            if n > 0:
                successful += 1
                total_snps += n
            else:
                empty += 1

        except Exception as e:

            print(
                f"ERROR: "
                f"{threshold} chr{chrom}: {e}"
            )

    summary.append({
        "Threshold": threshold,
        "Successful chromosomes": successful,
        "Empty files": empty,
        "Missing files": missing,
        "Total clumped SNPs": total_snps
    })

summary_df = pd.DataFrame(summary)

print(summary_df.to_string(index=False))

THRESHOLD-SPECIFIC CLUMPING AUDIT
Threshold  Successful chromosomes  Empty files  Missing files  Total clumped SNPs
   P5e-08                      21            0              1                 633
   P1e-05                      22            0              0                1636
  P0.0001                      22            0              0                3302
   P0.001                      22            0              0               11288
    P0.01                      22            0              0               74325
    P0.05                      22            0              0              358128


## 26. Construct PRS weight files

The index variants identified by LD clumping are extracted from the harmonized GWAS.

For each P-value threshold:

1. the index SNP identifiers are collected from the chromosome-specific `.clumps` files;
2. duplicate index SNPs are removed;
3. the SNP identifiers are matched back to the harmonized GWAS;
4. the corresponding effect allele, other allele, effect estimate, standard error, allele frequency and P-value are retained; and
5. the resulting variant table is saved as the PRS weight file.

The GWAS `BETA` value is used as the PRS weight.

The effect allele is the allele corresponding to the reported GWAS effect estimate.

### PRS calculation principle

For an individual *i*, the polygenic risk score is conceptually calculated as:

PRSᵢ = Σⱼ Gᵢⱼ × βⱼ

where:

- Gᵢⱼ is the dosage of the effect allele for variant *j* in individual *i*; and
- βⱼ is the GWAS effect estimate for variant *j*.

Actual individual-level PRS scoring is not performed in this notebook because an independent target genotype dataset is not yet available.

In [38]:
import os
import pandas as pd

base = "/content/drive/MyDrive/PRS_T2D_A-E/"

harmonized_file = (
    base +
    "results/harmonized/T2D_PheCode250_2_harmonized.tsv.gz"
)

clump_base = (
    base +
    "results/ld_clumping/threshold_clumps/"
)

weight_base = (
    base +
    "results/weights/"
)

thresholds = [
    "P5e-08",
    "P1e-05",
    "P0.0001",
    "P0.001",
    "P0.01",
    "P0.05"
]

# 1. Load harmonized AFRICAN GWAS

print("Loading harmonized African GWAS...")

gwas = pd.read_csv(
    harmonized_file,
    sep="\t",
    compression="gzip"
)

print("GWAS variants:", len(gwas))
print("GWAS columns:", gwas.columns.tolist())

# Keep fields required for PRS weights

gwas_weights = gwas[
    ["SNP", "CHR", "BP", "A1", "A2", "EAF", "BETA", "SE", "P", "N"]
].copy()

gwas_weights["SNP"] = (
    gwas_weights["SNP"]
    .astype(str)
    .str.strip()
)

# Remove duplicate SNP IDs

gwas_weights = gwas_weights.drop_duplicates(
    subset="SNP",
    keep="first"
)

# Index by SNP for fast matching

gwas_weights = gwas_weights.set_index("SNP")

print(
    "Unique African GWAS SNPs:",
    len(gwas_weights)
)


# 2. Process each P-value threshold

for threshold in thresholds:

    print(f"\nPROCESSING {threshold}")

    threshold_dir = (
        clump_base +
        threshold +
        "/"
    )

    all_clumped = []

    # Read chromosome-specific clump files

    for chrom in range(1, 23):

        clump_file = (
            threshold_dir +
            f"clump_chr{chrom}.clumps"
        )

        if not os.path.exists(clump_file):

            print(
                f"WARNING: missing chr{chrom} clump file"
            )

            continue

        clumps = pd.read_csv(
            clump_file,
            sep=r"\s+"
        )

        if "ID" not in clumps.columns:

            print(
                f"WARNING: ID column missing in chr{chrom}"
            )

            continue

        ids = (
            clumps["ID"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        all_clumped.extend(
            ids.tolist()
        )

    # Combine and remove duplicate index SNPs

    clumped_snps = (
        pd.Series(
            all_clumped,
            dtype="string"
        )
        .drop_duplicates()
    )

    print(
        "Unique clumped SNPs:",
        len(clumped_snps)
    )

    # Match index SNPs to AFRICAN harmonized GWAS

    matched = gwas_weights[
        gwas_weights.index.isin(clumped_snps)
    ].copy()

    print(
        "Matched to African harmonized GWAS:",
        len(matched)
    )

    # Matching rate

    if len(clumped_snps) > 0:

        match_rate = (
            len(matched) /
            len(clumped_snps)
        ) * 100

        print(
            f"Matching rate: {match_rate:.2f}%"
        )

    # Restore SNP as a column

    matched = matched.reset_index()

    # Save to threshold folder

    output_dir = (
        weight_base +
        threshold +
        "/"
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    output_file = (
        output_dir +
        "T2D_PheCode250_2_AFR_weights.tsv"
    )

    matched.to_csv(
        output_file,
        sep="\t",
        index=False
    )

    print("Saved:")
    print(output_file)

print(
    "\nAFR GWAS / EUR LD PRS WEIGHT "
    "CONSTRUCTION COMPLETE"
)

Loading harmonized African GWAS...
GWAS variants: 20948382
GWAS columns: ['SNP', 'CHR', 'BP', 'A1', 'A2', 'EAF', 'BETA', 'SE', 'P', 'N']
Unique African GWAS SNPs: 20948382

PROCESSING P5e-08
Unique clumped SNPs: 633
Matched to African harmonized GWAS: 633
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_A-E/results/weights/P5e-08/T2D_PheCode250_2_AFR_weights.tsv

PROCESSING P1e-05
Unique clumped SNPs: 1636
Matched to African harmonized GWAS: 1636
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_A-E/results/weights/P1e-05/T2D_PheCode250_2_AFR_weights.tsv

PROCESSING P0.0001
Unique clumped SNPs: 3302
Matched to African harmonized GWAS: 3302
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_A-E/results/weights/P0.0001/T2D_PheCode250_2_AFR_weights.tsv

PROCESSING P0.001
Unique clumped SNPs: 11288
Matched to African harmonized GWAS: 11288
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_A-E/results/weights/P0.001/T2D_PheCode250_2_AFR_weights.ts

## 27. Verify the final PRS weight files

The six final PRS weight files are verified.

For each threshold, the following are checked:

- file existence;
- number of variants;
- required columns;
- duplicate SNP identifiers;
- missing effect estimates;
- missing effect alleles; and
- successful matching of clumped index SNPs back to the harmonized GWAS.

This represents the final computational checkpoint of the discovery-stage PRS pipeline.

In [39]:
import os
import pandas as pd

base = "/content/drive/MyDrive/PRS_T2D_A-E/"

weight_base = (
    base +
    "results/weights/"
)

harmonized_file = (
    base +
    "results/harmonized/T2D_PheCode250_2_harmonized.tsv.gz"
)

thresholds = [
    "P5e-08",
    "P1e-05",
    "P0.0001",
    "P0.001",
    "P0.01",
    "P0.05"
]

required_columns = [
    "SNP",
    "CHR",
    "BP",
    "A1",
    "A2",
    "EAF",
    "BETA",
    "SE",
    "P",
    "N"
]

# Load harmonized African GWAS SNP IDs for matching verification

print("Loading harmonized African GWAS SNP IDs...")

harmonized_snps = set()

for chunk in pd.read_csv(
    harmonized_file,
    sep="\t",
    usecols=["SNP"],
    chunksize=500_000
):
    harmonized_snps.update(
        chunk["SNP"].dropna().astype(str)
    )

print(
    "Unique harmonized GWAS SNPs:",
    f"{len(harmonized_snps):,}"
)

print("\nVERIFYING FINAL A-E PRS WEIGHT FILES")

for threshold in thresholds:

    print(f"\n{threshold}")

    weight_file = (
        weight_base +
        threshold +
        "/T2D_PheCode250_2_AFR_weights.tsv"
    )

    # 1. File existence

    if not os.path.exists(weight_file):
        print("FAIL: weight file does not exist")
        continue

    print("File exists: PASS")

    # Load weight file

    weights = pd.read_csv(
        weight_file,
        sep="\t"
    )

    # 2. Number of variants

    print(
        "Number of variants:",
        f"{len(weights):,}"
    )

    # 3. Required columns

    missing_columns = [
        col for col in required_columns
        if col not in weights.columns
    ]

    if missing_columns:
        print(
            "Required columns: FAIL",
            missing_columns
        )
    else:
        print("Required columns: PASS")

    # Stop this threshold if essential columns are missing

    if any(
        col not in weights.columns
        for col in ["SNP", "A1", "BETA"]
    ):
        print("Skipping remaining checks because essential columns are missing")
        continue

    # 4. Duplicate SNP identifiers

    duplicate_count = weights["SNP"].duplicated().sum()

    if duplicate_count == 0:
        print("Duplicate SNP IDs: PASS")
    else:
        print(
            "Duplicate SNP IDs: FAIL",
            duplicate_count
        )

    # 5. Missing effect estimates

    missing_beta = weights["BETA"].isna().sum()

    if missing_beta == 0:
        print("Missing BETA values: PASS")
    else:
        print(
            "Missing BETA values: FAIL",
            missing_beta
        )

    # 6. Missing effect alleles

    missing_a1 = weights["A1"].isna().sum()

    if missing_a1 == 0:
        print("Missing effect alleles: PASS")
    else:
        print(
            "Missing effect alleles: FAIL",
            missing_a1
        )

    # 7. Match weight SNPs to harmonized African GWAS

    weight_snps = set(
        weights["SNP"]
        .dropna()
        .astype(str)
    )

    matched_snps = weight_snps.intersection(
        harmonized_snps
    )

    unmatched_count = (
        len(weight_snps) -
        len(matched_snps)
    )

    if unmatched_count == 0:
        print(
            "Matching to harmonized African GWAS: PASS"
        )
    else:
        print(
            "Matching to harmonized African GWAS: FAIL",
            unmatched_count,
            "unmatched SNPs"
        )

    # Matching rate

    if len(weight_snps) > 0:

        match_rate = (
            len(matched_snps) /
            len(weight_snps)
        ) * 100

        print(
            f"Matching rate: {match_rate:.2f}%"
        )

print("\nFINAL A-E PRS WEIGHT VERIFICATION COMPLETE")

Loading harmonized African GWAS SNP IDs...
Unique harmonized GWAS SNPs: 20,948,382

VERIFYING FINAL A-E PRS WEIGHT FILES

P5e-08
File exists: PASS
Number of variants: 633
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized African GWAS: PASS
Matching rate: 100.00%

P1e-05
File exists: PASS
Number of variants: 1,636
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized African GWAS: PASS
Matching rate: 100.00%

P0.0001
File exists: PASS
Number of variants: 3,302
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized African GWAS: PASS
Matching rate: 100.00%

P0.001
File exists: PASS
Number of variants: 11,288
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized African GWAS: PASS
Matching rate: 100.00%

P0

# Final stopping point

The discovery-stage PRS pipeline is complete.

The pipeline has produced verified PRS weight files at six GWAS P-value thresholds after:

1. GWAS inspection;
2. GWAS quality control;
3. GWAS harmonization;
4. candidate SNP selection;
5. construction of an African 1000 Genomes LD reference;
6. candidate SNP matching;
7. chromosome-specific reference construction;
8. chromosome-wise LD clumping; and
9. PRS weight construction.

The pipeline stops here because an independent target genotype and phenotype dataset is required for actual PRS scoring and validation.

No target-level PRS calculation, regression analysis, ROC-AUC analysis, calibration analysis, or predictive validation is performed in this notebook.

The End.